# Reproducible four-compartment FEM workflow

This notebook reproduces the endpoint-constrained, one-dimensional, four-compartment finite element modeling workflow used to analyze ketoconazole transport through porcine skin. It is designed to run from a local clone of this repository or from a Jupyter environment without Google Drive-specific paths.

Run all cells from top to bottom. Generated files are written to `outputs/models`, `outputs/tables`, `outputs/figures`, and `outputs/manuscript_ready_outputs`.


In [ ]:
# Repository setup: locate the project root and create output folders

from pathlib import Path
import os

def find_repository_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").exists() and (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter from the cloned repository or one of its subfolders."
    )

REPO_ROOT = find_repository_root()
PROJECT_DIR = str(REPO_ROOT)
DATA_PROCESSED_DIR = REPO_ROOT / "data" / "processed"
MODELS_DIR = REPO_ROOT / "outputs" / "models"
TABLES_DIR = REPO_ROOT / "outputs" / "tables"
FIGURES_DIR = REPO_ROOT / "outputs" / "figures"
MANUSCRIPT_DIR = REPO_ROOT / "outputs" / "manuscript_ready_outputs"

for folder in [DATA_PROCESSED_DIR, MODELS_DIR, TABLES_DIR, FIGURES_DIR, MANUSCRIPT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)

# Full nonlinear optimization can take a long time.
# False loads the archived fitted results and regenerates all downstream analyses.
# Set True only when intentionally refitting both formulations from all starting points.
RUN_FULL_OPTIMIZATION = False
print("Run full optimization:", RUN_FULL_OPTIMIZATION)


In [ ]:
# Cell 1: Project setup and experimental dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

# -----------------------------
# Experimental constants
# -----------------------------
# All amounts are expressed per diffusion area: ug/cm^2

AREA_CM2 = 2.2
RECEPTOR_VOLUME_ML = 6.5
SKIN_THICKNESS_CM = 0.09   # 0.9 mm = 0.09 cm, using midpoint of 0.8 to 1.0 mm
SC_THICKNESS_CM = 0.002    # 20 um = 0.002 cm, assumed SC thickness
EP_THICKNESS_CM = SKIN_THICKNESS_CM - SC_THICKNESS_CM

TEMPERATURE_C = 32.0

print("Experimental constants")
print("Area (cm^2):", AREA_CM2)
print("Receptor volume (mL):", RECEPTOR_VOLUME_ML)
print("Total skin thickness (cm):", SKIN_THICKNESS_CM)
print("SC thickness (cm):", SC_THICKNESS_CM)
print("EP/dermis thickness (cm):", EP_THICKNESS_CM)
print("Temperature (C):", TEMPERATURE_C)


# -----------------------------
# Endpoint experimental data from the outline
# -----------------------------
endpoint_data = pd.DataFrame([
    {
        "formulation": "Free KC",
        "receptor_24h_ug_cm2": 26.81,
        "SC_24h_ug_cm2": 0.90,
        "EP_24h_ug_cm2": 6.20
    },
    {
        "formulation": "KLZ-NPs",
        "receptor_24h_ug_cm2": 33.19,
        "SC_24h_ug_cm2": 2.70,
        "EP_24h_ug_cm2": 9.40
    }
])

endpoint_data["known_recovered_excluding_donor_ug_cm2"] = (
    endpoint_data["receptor_24h_ug_cm2"]
    + endpoint_data["SC_24h_ug_cm2"]
    + endpoint_data["EP_24h_ug_cm2"]
)

display(endpoint_data)


# -----------------------------
# Receptor-time data
# These are the porcine-skin receptor data used for fitting.
# If later we decide to refine digitized values, we will update only this table.
# -----------------------------
receptor_data = pd.DataFrame([
    # Free KC, Fig. 2B
    {"formulation": "Free KC", "time_h": 4,  "Q_receptor_ug_cm2": 0.70},
    {"formulation": "Free KC", "time_h": 6,  "Q_receptor_ug_cm2": 1.80},
    {"formulation": "Free KC", "time_h": 8,  "Q_receptor_ug_cm2": 4.30},
    {"formulation": "Free KC", "time_h": 12, "Q_receptor_ug_cm2": 10.40},
    {"formulation": "Free KC", "time_h": 24, "Q_receptor_ug_cm2": 26.81},

    # KLZ-NPs, Fig. 2B
    {"formulation": "KLZ-NPs", "time_h": 4,  "Q_receptor_ug_cm2": 1.70},
    {"formulation": "KLZ-NPs", "time_h": 6,  "Q_receptor_ug_cm2": 3.80},
    {"formulation": "KLZ-NPs", "time_h": 8,  "Q_receptor_ug_cm2": 6.70},
    {"formulation": "KLZ-NPs", "time_h": 12, "Q_receptor_ug_cm2": 13.60},
    {"formulation": "KLZ-NPs", "time_h": 24, "Q_receptor_ug_cm2": 33.19},
])

display(receptor_data)


# -----------------------------
# Skin retention data
# These intermediate values are kept for qualitative comparison only.
# The 24 h values are used as endpoint constraints.
# -----------------------------
skin_retention_data = pd.DataFrame([
    # Free KC, SC, Fig. 2D
    {"formulation": "Free KC", "compartment": "SC", "time_h": 0.5, "Q_ug_cm2": 3.10},
    {"formulation": "Free KC", "compartment": "SC", "time_h": 1,   "Q_ug_cm2": 3.30},
    {"formulation": "Free KC", "compartment": "SC", "time_h": 2,   "Q_ug_cm2": 2.60},
    {"formulation": "Free KC", "compartment": "SC", "time_h": 4,   "Q_ug_cm2": 1.20},
    {"formulation": "Free KC", "compartment": "SC", "time_h": 6,   "Q_ug_cm2": 1.20},
    {"formulation": "Free KC", "compartment": "SC", "time_h": 8,   "Q_ug_cm2": 0.70},
    {"formulation": "Free KC", "compartment": "SC", "time_h": 12,  "Q_ug_cm2": 0.80},
    {"formulation": "Free KC", "compartment": "SC", "time_h": 24,  "Q_ug_cm2": 0.90},

    # KLZ-NPs, SC, Fig. 2D
    {"formulation": "KLZ-NPs", "compartment": "SC", "time_h": 0.5, "Q_ug_cm2": 3.00},
    {"formulation": "KLZ-NPs", "compartment": "SC", "time_h": 1,   "Q_ug_cm2": 4.40},
    {"formulation": "KLZ-NPs", "compartment": "SC", "time_h": 2,   "Q_ug_cm2": 7.10},
    {"formulation": "KLZ-NPs", "compartment": "SC", "time_h": 4,   "Q_ug_cm2": 13.50},
    {"formulation": "KLZ-NPs", "compartment": "SC", "time_h": 6,   "Q_ug_cm2": 2.70},
    {"formulation": "KLZ-NPs", "compartment": "SC", "time_h": 8,   "Q_ug_cm2": 2.20},
    {"formulation": "KLZ-NPs", "compartment": "SC", "time_h": 12,  "Q_ug_cm2": 4.50},
    {"formulation": "KLZ-NPs", "compartment": "SC", "time_h": 24,  "Q_ug_cm2": 2.70},

    # Free KC, EP/dermis, Fig. 2F
    {"formulation": "Free KC", "compartment": "EP", "time_h": 0.5, "Q_ug_cm2": 2.20},
    {"formulation": "Free KC", "compartment": "EP", "time_h": 1,   "Q_ug_cm2": 3.60},
    {"formulation": "Free KC", "compartment": "EP", "time_h": 2,   "Q_ug_cm2": 4.30},
    {"formulation": "Free KC", "compartment": "EP", "time_h": 4,   "Q_ug_cm2": 6.60},
    {"formulation": "Free KC", "compartment": "EP", "time_h": 6,   "Q_ug_cm2": 4.80},
    {"formulation": "Free KC", "compartment": "EP", "time_h": 8,   "Q_ug_cm2": 2.90},
    {"formulation": "Free KC", "compartment": "EP", "time_h": 12,  "Q_ug_cm2": 3.00},
    {"formulation": "Free KC", "compartment": "EP", "time_h": 24,  "Q_ug_cm2": 6.20},

    # KLZ-NPs, EP/dermis, Fig. 2F
    {"formulation": "KLZ-NPs", "compartment": "EP", "time_h": 0.5, "Q_ug_cm2": 2.10},
    {"formulation": "KLZ-NPs", "compartment": "EP", "time_h": 1,   "Q_ug_cm2": 4.80},
    {"formulation": "KLZ-NPs", "compartment": "EP", "time_h": 2,   "Q_ug_cm2": 9.70},
    {"formulation": "KLZ-NPs", "compartment": "EP", "time_h": 4,   "Q_ug_cm2": 6.60},
    {"formulation": "KLZ-NPs", "compartment": "EP", "time_h": 6,   "Q_ug_cm2": 8.90},
    {"formulation": "KLZ-NPs", "compartment": "EP", "time_h": 8,   "Q_ug_cm2": 9.40},
    {"formulation": "KLZ-NPs", "compartment": "EP", "time_h": 12,  "Q_ug_cm2": 6.30},
    {"formulation": "KLZ-NPs", "compartment": "EP", "time_h": 24,  "Q_ug_cm2": 9.40},
])

display(skin_retention_data)


# -----------------------------
# Save cleaned input data to the repository
# -----------------------------

endpoint_path = f"{PROJECT_DIR}/data/processed/endpoint_constraints_24h.csv"
receptor_path = f"{PROJECT_DIR}/data/processed/receptor_time_data.csv"
retention_path = f"{PROJECT_DIR}/data/processed/skin_retention_data_qualitative.csv"

endpoint_data.to_csv(endpoint_path, index=False)
receptor_data.to_csv(receptor_path, index=False)
skin_retention_data.to_csv(retention_path, index=False)

print("Saved input data to the repository:")
print(endpoint_path)
print(receptor_path)
print(retention_path)

In [ ]:
# Cell 2: Plot experimental input data and save figure outputs

import os
import matplotlib.pyplot as plt

# -----------------------------
# Create figure
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel A: receptor permeation
ax = axes[0]

for formulation in ["Free KC", "KLZ-NPs"]:
    df = receptor_data[
        receptor_data["formulation"] == formulation
    ].sort_values("time_h")

    ax.plot(
        df["time_h"],
        df["Q_receptor_ug_cm2"],
        marker="o",
        linewidth=2,
        label=formulation
    )

ax.set_title("A. Receptor permeation")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Panel B: SC retention
ax = axes[1]

for formulation in ["Free KC", "KLZ-NPs"]:
    df = skin_retention_data[
        (skin_retention_data["formulation"] == formulation)
        & (skin_retention_data["compartment"] == "SC")
    ].sort_values("time_h")

    ax.plot(
        df["time_h"],
        df["Q_ug_cm2"],
        marker="o",
        linewidth=2,
        label=formulation
    )

ax.set_title("B. SC retention")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Panel C: EP/dermis retention
ax = axes[2]

for formulation in ["Free KC", "KLZ-NPs"]:
    df = skin_retention_data[
        (skin_retention_data["formulation"] == formulation)
        & (skin_retention_data["compartment"] == "EP")
    ].sort_values("time_h")

    ax.plot(
        df["time_h"],
        df["Q_ug_cm2"],
        marker="o",
        linewidth=2,
        label=formulation
    )

ax.set_title("C. EP/dermis retention")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# -----------------------------
# Save high-quality outputs to the repository
# -----------------------------
figure_dir = f"{PROJECT_DIR}/outputs/figures"

png_path = f"{figure_dir}/figure2_experimental_input_profiles.png"
pdf_path = f"{figure_dir}/figure2_experimental_input_profiles.pdf"
svg_path = f"{figure_dir}/figure2_experimental_input_profiles.svg"

plt.savefig(png_path, dpi=600, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")
plt.savefig(svg_path, bbox_inches="tight")

plt.show()

print("Figure files saved to the repository:")
print(png_path)
print(pdf_path)
print(svg_path)

# -----------------------------
# -----------------------------


In [ ]:
# Cell 3 revised: Build clean master tables for FEM modeling

import pandas as pd
import os

# -----------------------------
# 1) Receptor data
# Receptor-time profile is the primary dynamic fitting target.
# -----------------------------
receptor_df = receptor_data.copy()

receptor_df = receptor_df.rename(columns={
    "Q_receptor_ug_cm2": "observed_ug_cm2"
})

receptor_df["compartment"] = "R"
receptor_df["data_type"] = "receptor_time_profile"
receptor_df["use_for_fitting"] = True

receptor_df = receptor_df[
    ["formulation", "compartment", "time_h", "observed_ug_cm2", "data_type", "use_for_fitting"]
].sort_values(["formulation", "time_h"]).reset_index(drop=True)


# -----------------------------
# 2) Skin retention data
# Intermediate SC and EP values are kept for qualitative comparison only.
# They are NOT used as strict fitting targets.
# -----------------------------
skin_profile_df = skin_retention_data.copy()

skin_profile_df = skin_profile_df.rename(columns={
    "Q_ug_cm2": "observed_ug_cm2"
})

skin_profile_df["data_type"] = "skin_time_profile_qualitative"
skin_profile_df["use_for_fitting"] = False

skin_profile_df = skin_profile_df[
    ["formulation", "compartment", "time_h", "observed_ug_cm2", "data_type", "use_for_fitting"]
].sort_values(["formulation", "compartment", "time_h"]).reset_index(drop=True)


# -----------------------------
# 3) Skin endpoint constraints at 24 h
# Only SC and EP endpoints are added as fitting constraints.
# Receptor at 24 h is already included in receptor_df.
# -----------------------------
skin_endpoint_rows = []

for _, row in endpoint_data.iterrows():
    skin_endpoint_rows.append({
        "formulation": row["formulation"],
        "compartment": "SC",
        "time_h": 24.0,
        "observed_ug_cm2": row["SC_24h_ug_cm2"],
        "data_type": "skin_endpoint_24h",
        "use_for_fitting": True
    })

    skin_endpoint_rows.append({
        "formulation": row["formulation"],
        "compartment": "EP",
        "time_h": 24.0,
        "observed_ug_cm2": row["EP_24h_ug_cm2"],
        "data_type": "skin_endpoint_24h",
        "use_for_fitting": True
    })

skin_endpoint_df = pd.DataFrame(skin_endpoint_rows)

skin_endpoint_df = skin_endpoint_df.sort_values(
    ["formulation", "compartment", "time_h"]
).reset_index(drop=True)


# -----------------------------
# 4) Full profile dataset
# This is for visualization and qualitative comparison.
# -----------------------------
full_profile_df = pd.concat(
    [receptor_df, skin_profile_df],
    ignore_index=True
).sort_values(["formulation", "compartment", "time_h"]).reset_index(drop=True)


# -----------------------------
# 5) FEM fitting dataset
# This is the ONLY dataset used for parameter fitting.
# -----------------------------
fem_fit_data = pd.concat(
    [receptor_df, skin_endpoint_df],
    ignore_index=True
).sort_values(["formulation", "compartment", "time_h", "data_type"]).reset_index(drop=True)


# -----------------------------
# 6) Save processed tables to the repository
# -----------------------------
processed_dir = f"{PROJECT_DIR}/data/processed"

receptor_df.to_csv(f"{processed_dir}/receptor_profile_clean.csv", index=False)
skin_profile_df.to_csv(f"{processed_dir}/skin_retention_profile_qualitative.csv", index=False)
skin_endpoint_df.to_csv(f"{processed_dir}/skin_endpoint_constraints_24h.csv", index=False)
full_profile_df.to_csv(f"{processed_dir}/full_profile_data_clean.csv", index=False)
fem_fit_data.to_csv(f"{processed_dir}/fem_fit_data_master.csv", index=False)


# -----------------------------
# 7) Display quick check
# -----------------------------
print("receptor_df shape:", receptor_df.shape)
print("skin_profile_df shape:", skin_profile_df.shape)
print("skin_endpoint_df shape:", skin_endpoint_df.shape)
print("full_profile_df shape:", full_profile_df.shape)
print("fem_fit_data shape:", fem_fit_data.shape)

print("\nFitting data only:")
display(fem_fit_data)

print("\nQualitative skin profile data only:")
display(skin_profile_df.head(12))

In [ ]:
# Cell 4: Build four-layer FEM geometry and mesh

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Equivalent compartment thicknesses
# -----------------------------
# For donor and receptor, thickness is volume per diffusion area.
# 1 mL = 1 cm^3, so mL/cm^2 is equivalent to cm.

DONOR_VOLUME_ML = 1.0

L1_DONOR_CM = DONOR_VOLUME_ML / AREA_CM2
L2_SC_CM = SC_THICKNESS_CM
L3_EP_CM = EP_THICKNESS_CM
L4_RECEPTOR_CM = RECEPTOR_VOLUME_ML / AREA_CM2

# -----------------------------
# Number of elements per layer
# -----------------------------
# More elements are used in skin layers because these are the main transport barriers.

N1_DONOR = 20
N2_SC = 10
N3_EP = 100
N4_RECEPTOR = 30

layer_geometry = pd.DataFrame([
    {
        "layer_id": 1,
        "layer_name": "Donor compartment",
        "short_name": "Donor",
        "concentration_symbol": "C1",
        "diffusion_symbol": "D1",
        "length_cm": L1_DONOR_CM,
        "n_elements": N1_DONOR,
        "role": "Computational donor compartment"
    },
    {
        "layer_id": 2,
        "layer_name": "Stratum corneum",
        "short_name": "SC",
        "concentration_symbol": "C2",
        "diffusion_symbol": "D2",
        "length_cm": L2_SC_CM,
        "n_elements": N2_SC,
        "role": "Main barrier layer"
    },
    {
        "layer_id": 3,
        "layer_name": "Viable epidermis/dermis",
        "short_name": "EP/dermis",
        "concentration_symbol": "C3",
        "diffusion_symbol": "D3",
        "length_cm": L3_EP_CM,
        "n_elements": N3_EP,
        "role": "Deeper skin retention region"
    },
    {
        "layer_id": 4,
        "layer_name": "Receptor compartment",
        "short_name": "Receptor",
        "concentration_symbol": "C4",
        "diffusion_symbol": "D4",
        "length_cm": L4_RECEPTOR_CM,
        "n_elements": N4_RECEPTOR,
        "role": "Computational receptor compartment"
    }
])

layer_geometry["length_um"] = layer_geometry["length_cm"] * 10000.0
layer_geometry["start_cm"] = layer_geometry["length_cm"].cumsum() - layer_geometry["length_cm"]
layer_geometry["end_cm"] = layer_geometry["length_cm"].cumsum()

L_TOTAL_MODEL_CM = layer_geometry["length_cm"].sum()

print("Four-layer geometry")
display(layer_geometry)

print("Total computational model length (cm):", L_TOTAL_MODEL_CM)
print("Donor equivalent thickness (cm):", L1_DONOR_CM)
print("Receptor equivalent thickness (cm):", L4_RECEPTOR_CM)

# -----------------------------
# Build separate-node mesh for each layer
# -----------------------------
# We keep separate nodes at interfaces because later the model will allow
# partition jumps between adjacent layers.

node_rows = []
element_rows = []

global_node_id = 0
global_element_id = 0

for _, layer in layer_geometry.iterrows():
    layer_id = int(layer["layer_id"])
    short_name = layer["short_name"]
    length_cm = float(layer["length_cm"])
    n_elements = int(layer["n_elements"])
    start_cm = float(layer["start_cm"])

    x_local = np.linspace(0.0, length_cm, n_elements + 1)
    x_global = start_cm + x_local

    layer_node_ids = []

    for local_node_id, (xl, xg) in enumerate(zip(x_local, x_global)):
        node_rows.append({
            "global_node_id": global_node_id,
            "layer_id": layer_id,
            "layer_name": short_name,
            "local_node_id": local_node_id,
            "x_local_cm": xl,
            "x_global_cm": xg,
            "is_left_boundary": local_node_id == 0,
            "is_right_boundary": local_node_id == n_elements
        })
        layer_node_ids.append(global_node_id)
        global_node_id += 1

    for local_element_id in range(n_elements):
        element_rows.append({
            "global_element_id": global_element_id,
            "layer_id": layer_id,
            "layer_name": short_name,
            "local_element_id": local_element_id,
            "left_node": layer_node_ids[local_element_id],
            "right_node": layer_node_ids[local_element_id + 1],
            "x_left_cm": x_global[local_element_id],
            "x_right_cm": x_global[local_element_id + 1],
            "element_length_cm": x_global[local_element_id + 1] - x_global[local_element_id]
        })
        global_element_id += 1

mesh_nodes_df = pd.DataFrame(node_rows)
mesh_elements_df = pd.DataFrame(element_rows)

print("Mesh summary")
print("Total nodes:", len(mesh_nodes_df))
print("Total elements:", len(mesh_elements_df))

print("\nFirst nodes:")
display(mesh_nodes_df.head())

print("\nLast nodes:")
display(mesh_nodes_df.tail())

print("\nFirst elements:")
display(mesh_elements_df.head())

# -----------------------------
# Save geometry and mesh tables
# -----------------------------
tables_dir = f"{PROJECT_DIR}/outputs/tables"
processed_dir = f"{PROJECT_DIR}/data/processed"

layer_geometry_path = f"{tables_dir}/table4_four_layer_geometry.csv"
mesh_nodes_path = f"{processed_dir}/four_layer_mesh_nodes.csv"
mesh_elements_path = f"{processed_dir}/four_layer_mesh_elements.csv"

layer_geometry.to_csv(layer_geometry_path, index=False)
mesh_nodes_df.to_csv(mesh_nodes_path, index=False)
mesh_elements_df.to_csv(mesh_elements_path, index=False)

print("Saved geometry and mesh tables:")
print(layer_geometry_path)
print(mesh_nodes_path)
print(mesh_elements_path)

# -----------------------------
# Plot full computational geometry
# -----------------------------
fig, ax = plt.subplots(figsize=(12, 2.8))

ax.plot(mesh_nodes_df["x_global_cm"], np.zeros(len(mesh_nodes_df)), marker="o", linestyle="none", markersize=3)

for _, layer in layer_geometry.iterrows():
    ax.axvline(layer["start_cm"], linestyle="--", linewidth=1)
    ax.axvline(layer["end_cm"], linestyle="--", linewidth=1)

    x_mid = 0.5 * (layer["start_cm"] + layer["end_cm"])
    ax.text(x_mid, 0.08, layer["short_name"], ha="center", va="bottom")

ax.set_title("Four-layer computational FEM geometry")
ax.set_xlabel("Position across computational domain (cm)")
ax.set_yticks([])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

plt.tight_layout()

full_png = f"{PROJECT_DIR}/outputs/figures/figure_geometry_full_computational_domain.png"
full_pdf = f"{PROJECT_DIR}/outputs/figures/figure_geometry_full_computational_domain.pdf"
full_svg = f"{PROJECT_DIR}/outputs/figures/figure_geometry_full_computational_domain.svg"

plt.savefig(full_png, dpi=600, bbox_inches="tight")
plt.savefig(full_pdf, bbox_inches="tight")
plt.savefig(full_svg, bbox_inches="tight")

plt.show()

# -----------------------------
# Plot skin-domain zoom
# -----------------------------
skin_start_cm = layer_geometry.loc[layer_geometry["short_name"] == "SC", "start_cm"].iloc[0]
skin_end_cm = layer_geometry.loc[layer_geometry["short_name"] == "EP/dermis", "end_cm"].iloc[0]

skin_nodes = mesh_nodes_df[
    (mesh_nodes_df["x_global_cm"] >= skin_start_cm)
    & (mesh_nodes_df["x_global_cm"] <= skin_end_cm)
].copy()

fig, ax = plt.subplots(figsize=(12, 2.8))

ax.plot(skin_nodes["x_global_cm"], np.zeros(len(skin_nodes)), marker="o", linestyle="none", markersize=3)

for _, layer in layer_geometry[layer_geometry["layer_id"].isin([2, 3])].iterrows():
    ax.axvline(layer["start_cm"], linestyle="--", linewidth=1)
    ax.axvline(layer["end_cm"], linestyle="--", linewidth=1)

    x_mid = 0.5 * (layer["start_cm"] + layer["end_cm"])
    ax.text(x_mid, 0.08, layer["short_name"], ha="center", va="bottom")

ax.set_title("Skin-domain FEM mesh zoom: SC and EP/dermis")
ax.set_xlabel("Position across computational domain (cm)")
ax.set_yticks([])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

plt.tight_layout()

zoom_png = f"{PROJECT_DIR}/outputs/figures/figure_geometry_skin_domain_zoom.png"
zoom_pdf = f"{PROJECT_DIR}/outputs/figures/figure_geometry_skin_domain_zoom.pdf"
zoom_svg = f"{PROJECT_DIR}/outputs/figures/figure_geometry_skin_domain_zoom.svg"

plt.savefig(zoom_png, dpi=600, bbox_inches="tight")
plt.savefig(zoom_pdf, bbox_inches="tight")
plt.savefig(zoom_svg, bbox_inches="tight")

plt.show()

print("Saved geometry figures:")
print(full_png)
print(full_pdf)
print(full_svg)
print(zoom_png)
print(zoom_pdf)
print(zoom_svg)

# -----------------------------
# Download figures to local computer
# -----------------------------


In [ ]:
# Cell 5: Assemble FEM mass and diffusion matrices

import numpy as np
import pandas as pd
import os

# -----------------------------
# Required variable check
# -----------------------------
required_names = [
    "mesh_nodes_df",
    "mesh_elements_df",
    "layer_geometry",
    "PROJECT_DIR"
]

for name in required_names:
    if name not in globals():
        raise NameError(f"{name} is missing. Please run previous cells first.")

# -----------------------------
# Fixed large diffusion coefficients for well-mixed computational compartments
# -----------------------------
# These are not fitted skin parameters.
# They are used to make donor and receptor compartments nearly well-mixed.

D1_FIXED_CM2_H = 10.0
D4_FIXED_CM2_H = 10.0

# Test values for skin-layer diffusion coefficients.
# These are only for matrix testing, not final fitted values.
D2_TEST_CM2_H = 1e-6
D3_TEST_CM2_H = 1e-4

print("Fixed and test diffusion coefficients")
print("D1 donor fixed (cm2/h):", D1_FIXED_CM2_H)
print("D2 SC test (cm2/h):", D2_TEST_CM2_H)
print("D3 EP/dermis test (cm2/h):", D3_TEST_CM2_H)
print("D4 receptor fixed (cm2/h):", D4_FIXED_CM2_H)


# -----------------------------
# Helper maps
# -----------------------------
N_TOTAL_NODES = len(mesh_nodes_df)
N_TOTAL_ELEMENTS = len(mesh_elements_df)

layer_node_ids = {}

for layer_id in sorted(mesh_nodes_df["layer_id"].unique()):
    node_ids = mesh_nodes_df.loc[
        mesh_nodes_df["layer_id"] == layer_id,
        "global_node_id"
    ].astype(int).tolist()

    layer_node_ids[int(layer_id)] = node_ids

boundary_nodes = {}

for layer_id, node_ids in layer_node_ids.items():
    boundary_nodes[layer_id] = {
        "left": node_ids[0],
        "right": node_ids[-1]
    }

print("\nNode and element count")
print("Total nodes:", N_TOTAL_NODES)
print("Total elements:", N_TOTAL_ELEMENTS)

print("\nBoundary nodes by layer")
print(boundary_nodes)


# -----------------------------
# FEM matrix assembly function
# -----------------------------
def assemble_fem_matrices(D1, D2, D3, D4):
    """
    Assemble global FEM mass matrix and diffusion stiffness matrix.

    Units:
    D values: cm2/h
    x values: cm
    concentration: ug/cm3
    integrated amount: ug/cm2
    """

    D_by_layer = {
        1: D1,
        2: D2,
        3: D3,
        4: D4
    }

    M = np.zeros((N_TOTAL_NODES, N_TOTAL_NODES), dtype=float)
    K = np.zeros((N_TOTAL_NODES, N_TOTAL_NODES), dtype=float)

    for _, elem in mesh_elements_df.iterrows():
        layer_id = int(elem["layer_id"])
        left_node = int(elem["left_node"])
        right_node = int(elem["right_node"])
        h = float(elem["element_length_cm"])
        D = float(D_by_layer[layer_id])

        # Linear 1D finite element mass matrix
        M_e = (h / 6.0) * np.array([
            [2.0, 1.0],
            [1.0, 2.0]
        ])

        # Linear 1D finite element diffusion stiffness matrix
        K_e = (D / h) * np.array([
            [1.0, -1.0],
            [-1.0, 1.0]
        ])

        node_pair = [left_node, right_node]

        for a in range(2):
            for b in range(2):
                M[node_pair[a], node_pair[b]] += M_e[a, b]
                K[node_pair[a], node_pair[b]] += K_e[a, b]

    return M, K


# -----------------------------
# Assemble test matrices
# -----------------------------
M_test, K_test = assemble_fem_matrices(
    D1=D1_FIXED_CM2_H,
    D2=D2_TEST_CM2_H,
    D3=D3_TEST_CM2_H,
    D4=D4_FIXED_CM2_H
)

print("\nMatrix shapes")
print("M_test shape:", M_test.shape)
print("K_test shape:", K_test.shape)

# -----------------------------
# Matrix sanity checks
# -----------------------------
M_is_symmetric = np.allclose(M_test, M_test.T, rtol=1e-10, atol=1e-12)
K_is_symmetric = np.allclose(K_test, K_test.T, rtol=1e-10, atol=1e-12)

M_diag_positive = np.all(np.diag(M_test) > 0)
K_diag_nonnegative = np.all(np.diag(K_test) >= 0)

print("\nSanity checks")
print("Mass matrix symmetric:", M_is_symmetric)
print("Diffusion matrix symmetric:", K_is_symmetric)
print("Mass matrix diagonal positive:", M_diag_positive)
print("Diffusion matrix diagonal nonnegative:", K_diag_nonnegative)


# -----------------------------
# Function to integrate amount in each layer
# -----------------------------
def compute_layer_amounts(C):
    """
    Compute integrated amount per area in each layer from nodal concentration.

    Input:
    C: nodal concentration vector, ug/cm3

    Output:
    DataFrame with layer amounts, ug/cm2
    """

    if len(C) != N_TOTAL_NODES:
        raise ValueError("Concentration vector length does not match total number of nodes.")

    amount_by_layer = {
        1: 0.0,
        2: 0.0,
        3: 0.0,
        4: 0.0
    }

    for _, elem in mesh_elements_df.iterrows():
        layer_id = int(elem["layer_id"])
        left_node = int(elem["left_node"])
        right_node = int(elem["right_node"])
        h = float(elem["element_length_cm"])

        c_left = C[left_node]
        c_right = C[right_node]

        # Trapezoidal integration over the element
        amount_by_layer[layer_id] += 0.5 * h * (c_left + c_right)

    rows = []

    for layer_id, amount in amount_by_layer.items():
        layer_row = layer_geometry[layer_geometry["layer_id"] == layer_id].iloc[0]

        rows.append({
            "layer_id": layer_id,
            "short_name": layer_row["short_name"],
            "computed_amount_ug_cm2": amount,
            "expected_if_C_equals_1": layer_row["length_cm"]
        })

    return pd.DataFrame(rows)


# -----------------------------
# Integration sanity check
# -----------------------------
# If concentration is 1 ug/cm3 everywhere, integrated amount should equal layer thickness in cm.
C_uniform = np.ones(N_TOTAL_NODES)

amount_check_df = compute_layer_amounts(C_uniform)
amount_check_df["absolute_error"] = (
    amount_check_df["computed_amount_ug_cm2"]
    - amount_check_df["expected_if_C_equals_1"]
).abs()

print("\nLayer integration check")
display(amount_check_df)

max_integration_error = amount_check_df["absolute_error"].max()
print("Maximum integration error:", max_integration_error)


# -----------------------------
# Save sanity check table
# -----------------------------
tables_dir = f"{PROJECT_DIR}/outputs/tables"
matrix_check_path = f"{tables_dir}/fem_matrix_and_integration_sanity_check.csv"

amount_check_df.to_csv(matrix_check_path, index=False)

print("\nSaved matrix sanity check table:")
print(matrix_check_path)

In [ ]:
# Cell 6: Add interface fluxes and test four-layer simulation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# -----------------------------
# Required variable check
# -----------------------------
required_names = [
    "assemble_fem_matrices",
    "compute_layer_amounts",
    "boundary_nodes",
    "layer_geometry",
    "N_TOTAL_NODES",
    "L1_DONOR_CM",
    "PROJECT_DIR"
]

for name in required_names:
    if name not in globals():
        raise NameError(f"{name} is missing. Please run previous cells first.")

# -----------------------------
# Fixed interface assumptions
# -----------------------------
# K12 and K23 are fixed large to approximate quasi-equilibrium.
# These are not fitted in Case 1 or Case 2.

K12_FIXED_CM_H = 100.0
K23_FIXED_CM_H = 100.0

print("Fixed interface mass transfer coefficients")
print("K12 donor-SC fixed (cm/h):", K12_FIXED_CM_H)
print("K23 SC-EP fixed (cm/h):", K23_FIXED_CM_H)


# -----------------------------
# Interface flux function
# -----------------------------
def compute_interface_fluxes(C, P12, P23, P34, K34):
    """
    Compute interfacial fluxes between adjacent layers.

    Positive flux means transport from left to right:
    donor -> SC -> EP/dermis -> receptor

    Units:
    C: ug/cm3
    K: cm/h
    J: ug/cm2/h
    """

    # Boundary node concentrations
    C1_right = C[boundary_nodes[1]["right"]]
    C2_left = C[boundary_nodes[2]["left"]]

    C2_right = C[boundary_nodes[2]["right"]]
    C3_left = C[boundary_nodes[3]["left"]]

    C3_right = C[boundary_nodes[3]["right"]]
    C4_left = C[boundary_nodes[4]["left"]]

    # Interface fluxes
    J12 = K12_FIXED_CM_H * ((C1_right / P12) - C2_left)
    J23 = K23_FIXED_CM_H * ((C2_right / P23) - C3_left)
    J34 = K34 * ((C3_right / P34) - C4_left)

    return J12, J23, J34


# -----------------------------
# Four-layer ODE system
# -----------------------------
def four_layer_ode_system(t, C, M, K, P12, P23, P34, K34):
    """
    Semi-discrete FEM ODE system for the four-layer model.

    M dC/dt = -K C + interface flux terms
    """

    rhs = -K @ C

    J12, J23, J34 = compute_interface_fluxes(
        C=C,
        P12=P12,
        P23=P23,
        P34=P34,
        K34=K34
    )

    # Interface 12: donor loses, SC gains
    rhs[boundary_nodes[1]["right"]] -= J12
    rhs[boundary_nodes[2]["left"]] += J12

    # Interface 23: SC loses, EP gains
    rhs[boundary_nodes[2]["right"]] -= J23
    rhs[boundary_nodes[3]["left"]] += J23

    # Interface 34: EP loses, receptor gains
    rhs[boundary_nodes[3]["right"]] -= J34
    rhs[boundary_nodes[4]["left"]] += J34

    dCdt = np.linalg.solve(M, rhs)

    return dCdt


# -----------------------------
# Simulation function
# -----------------------------
def simulate_four_layer_model(
    times_h,
    Q0_eff_ug_cm2,
    D2_cm2_h,
    D3_cm2_h,
    P12,
    P23,
    P34,
    K34_cm_h,
    D1_cm2_h=D1_FIXED_CM2_H,
    D4_cm2_h=D4_FIXED_CM2_H
):
    """
    Simulate the four-layer FEM model and return compartment amounts.

    Q0_eff_ug_cm2 is converted into a uniform donor concentration at t = 0.
    """

    times_h = np.asarray(times_h, dtype=float)
    times_h = np.sort(np.unique(times_h))

    if times_h[0] < 0:
        raise ValueError("Simulation times must be nonnegative.")

    M, K = assemble_fem_matrices(
        D1=D1_cm2_h,
        D2=D2_cm2_h,
        D3=D3_cm2_h,
        D4=D4_cm2_h
    )

    # Initial concentration vector
    C0 = np.zeros(N_TOTAL_NODES, dtype=float)

    # Convert donor amount per area to donor concentration.
    # If donor concentration is uniform:
    # Q0_eff = C0_donor * L1_DONOR_CM
    C0_donor = Q0_eff_ug_cm2 / L1_DONOR_CM

    donor_nodes = mesh_nodes_df.loc[
        mesh_nodes_df["layer_id"] == 1,
        "global_node_id"
    ].astype(int).tolist()

    C0[donor_nodes] = C0_donor

    sol = solve_ivp(
        fun=lambda t, C: four_layer_ode_system(
            t=t,
            C=C,
            M=M,
            K=K,
            P12=P12,
            P23=P23,
            P34=P34,
            K34=K34_cm_h
        ),
        t_span=(0.0, float(times_h.max())),
        y0=C0,
        t_eval=times_h,
        method="LSODA",
        rtol=1e-7,
        atol=1e-9
    )

    if not sol.success:
        raise RuntimeError(f"Simulation failed: {sol.message}")

    amount_rows = []

    for idx, t in enumerate(sol.t):
        C_t = sol.y[:, idx]
        amount_df = compute_layer_amounts(C_t)

        Q1 = amount_df.loc[amount_df["layer_id"] == 1, "computed_amount_ug_cm2"].iloc[0]
        Q2 = amount_df.loc[amount_df["layer_id"] == 2, "computed_amount_ug_cm2"].iloc[0]
        Q3 = amount_df.loc[amount_df["layer_id"] == 3, "computed_amount_ug_cm2"].iloc[0]
        Q4 = amount_df.loc[amount_df["layer_id"] == 4, "computed_amount_ug_cm2"].iloc[0]

        amount_rows.append({
            "time_h": t,
            "Q1_donor_ug_cm2": Q1,
            "Q2_SC_ug_cm2": Q2,
            "Q3_EP_ug_cm2": Q3,
            "Q4_receptor_ug_cm2": Q4,
            "Q_total_ug_cm2": Q1 + Q2 + Q3 + Q4
        })

    amount_time_df = pd.DataFrame(amount_rows)

    return amount_time_df, sol


# -----------------------------
# Test simulation
# -----------------------------
test_times = [0, 1, 2, 4, 6, 8, 12, 24]

test_params = {
    "Q0_eff_ug_cm2": 50.0,
    "D2_cm2_h": 1e-6,
    "D3_cm2_h": 1e-4,
    "P12": 1.0,
    "P23": 1.0,
    "P34": 1.0,
    "K34_cm_h": 0.1
}

test_amounts_df, test_solution = simulate_four_layer_model(
    times_h=test_times,
    **test_params
)

print("Test simulation completed.")
display(test_amounts_df)

# -----------------------------
# Mass conservation check
# -----------------------------
initial_mass = test_amounts_df["Q_total_ug_cm2"].iloc[0]
final_mass = test_amounts_df["Q_total_ug_cm2"].iloc[-1]
mass_error = final_mass - initial_mass

print("Initial total amount (ug/cm2):", initial_mass)
print("Final total amount (ug/cm2):", final_mass)
print("Mass conservation error (ug/cm2):", mass_error)


# -----------------------------
# Save test simulation output
# -----------------------------
models_dir = f"{PROJECT_DIR}/outputs/models"
figures_dir = f"{PROJECT_DIR}/outputs/figures"

test_csv_path = f"{models_dir}/test_four_layer_interface_simulation.csv"
test_amounts_df.to_csv(test_csv_path, index=False)

print("Saved test simulation table:")
print(test_csv_path)


# -----------------------------
# Plot test compartment amounts
# -----------------------------
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(test_amounts_df["time_h"], test_amounts_df["Q1_donor_ug_cm2"], marker="o", label="Donor")
ax.plot(test_amounts_df["time_h"], test_amounts_df["Q2_SC_ug_cm2"], marker="o", label="SC")
ax.plot(test_amounts_df["time_h"], test_amounts_df["Q3_EP_ug_cm2"], marker="o", label="EP/dermis")
ax.plot(test_amounts_df["time_h"], test_amounts_df["Q4_receptor_ug_cm2"], marker="o", label="Receptor")
ax.plot(test_amounts_df["time_h"], test_amounts_df["Q_total_ug_cm2"], marker="o", linestyle="--", label="Total")

ax.set_title("Test four-layer FEM simulation with interface fluxes")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

test_png_path = f"{figures_dir}/test_four_layer_interface_simulation.png"
test_pdf_path = f"{figures_dir}/test_four_layer_interface_simulation.pdf"
test_svg_path = f"{figures_dir}/test_four_layer_interface_simulation.svg"

plt.savefig(test_png_path, dpi=600, bbox_inches="tight")
plt.savefig(test_pdf_path, bbox_inches="tight")
plt.savefig(test_svg_path, bbox_inches="tight")

plt.show()

print("Saved test simulation figures:")
print(test_png_path)
print(test_pdf_path)
print(test_svg_path)



In [ ]:
# Cell 7: Prediction and residual functions for Case 1

import numpy as np
import pandas as pd
import os

# -----------------------------
# Required variable check
# -----------------------------
required_names = [
    "simulate_four_layer_model",
    "fem_fit_data",
    "endpoint_data",
    "PROJECT_DIR"
]

for name in required_names:
    if name not in globals():
        raise NameError(f"{name} is missing. Please run previous cells first.")


# -----------------------------
# Case 1 parameter definition
# -----------------------------
# Case 1 fitted parameters:
# Q0_eff, D2, D3, P12, P34, K34
#
# Fixed:
# P23 = 1
# K12 and K23 are already fixed large in Cell 6.
# D1 and D4 are already fixed large.

CASE1_PARAMETER_NAMES = [
    "Q0_eff_ug_cm2",
    "D2_SC_cm2_h",
    "D3_EP_cm2_h",
    "P12_donor_SC",
    "P34_EP_receptor",
    "K34_cm_h"
]

print("Case 1 fitted parameters:")
for name in CASE1_PARAMETER_NAMES:
    print("-", name)


# -----------------------------
# Convert log-parameters to real positive parameters
# -----------------------------
def unpack_case1_log_params(log_params):
    """
    Convert log-transformed Case 1 parameters into real positive parameters.
    """
    params = np.exp(np.asarray(log_params, dtype=float))

    return {
        "Q0_eff_ug_cm2": params[0],
        "D2_SC_cm2_h": params[1],
        "D3_EP_cm2_h": params[2],
        "P12_donor_SC": params[3],
        "P34_EP_receptor": params[4],
        "K34_cm_h": params[5]
    }


# -----------------------------
# Simulate Case 1 for one formulation
# -----------------------------
def simulate_case1_for_formulation(log_params, formulation_name, fit_data):
    """
    Run Case 1 model for one formulation and return model-predicted amounts.
    """
    p = unpack_case1_log_params(log_params)

    df_form = fit_data[
        fit_data["formulation"] == formulation_name
    ].copy()

    times = sorted(df_form["time_h"].unique())

    sim_df, sol = simulate_four_layer_model(
        times_h=times,
        Q0_eff_ug_cm2=p["Q0_eff_ug_cm2"],
        D2_cm2_h=p["D2_SC_cm2_h"],
        D3_cm2_h=p["D3_EP_cm2_h"],
        P12=p["P12_donor_SC"],
        P23=1.0,
        P34=p["P34_EP_receptor"],
        K34_cm_h=p["K34_cm_h"]
    )

    return sim_df, sol, p


# -----------------------------
# Prediction for each observed row
# -----------------------------
def predict_case1_rows(log_params, formulation_name, fit_data):
    """
    Predict model values for all fitting rows of one formulation.
    """
    sim_df, sol, p = simulate_case1_for_formulation(
        log_params=log_params,
        formulation_name=formulation_name,
        fit_data=fit_data
    )

    sim_indexed = sim_df.set_index("time_h")

    df_form = fit_data[
        fit_data["formulation"] == formulation_name
    ].copy()

    predictions = []

    for _, row in df_form.iterrows():
        t = float(row["time_h"])
        compartment = row["compartment"]

        if compartment == "R":
            pred = sim_indexed.loc[t, "Q4_receptor_ug_cm2"]
        elif compartment == "SC":
            pred = sim_indexed.loc[t, "Q2_SC_ug_cm2"]
        elif compartment == "EP":
            pred = sim_indexed.loc[t, "Q3_EP_ug_cm2"]
        else:
            raise ValueError(f"Unknown compartment: {compartment}")

        predictions.append(pred)

    df_pred = df_form.copy()
    df_pred["model_ug_cm2"] = predictions
    df_pred["error_ug_cm2"] = df_pred["model_ug_cm2"] - df_pred["observed_ug_cm2"]

    return df_pred, sim_df, p


# -----------------------------
# Normalization scales for objective function
# -----------------------------
def get_case1_scales(formulation_name, fit_data):
    """
    Get normalization scales for receptor, SC endpoint, and EP endpoint.
    """
    df_form = fit_data[
        fit_data["formulation"] == formulation_name
    ].copy()

    receptor_scale = df_form[
        df_form["compartment"] == "R"
    ]["observed_ug_cm2"].max()

    sc_scale = df_form[
        df_form["compartment"] == "SC"
    ]["observed_ug_cm2"].max()

    ep_scale = df_form[
        df_form["compartment"] == "EP"
    ]["observed_ug_cm2"].max()

    scales = {
        "R": max(receptor_scale, 1e-9),
        "SC": max(sc_scale, 1e-9),
        "EP": max(ep_scale, 1e-9)
    }

    return scales


# -----------------------------
# Residual function for least-squares fitting
# -----------------------------
def residuals_case1(log_params, formulation_name, fit_data, weights=None):
    """
    Compute normalized residual vector for Case 1.
    """
    if weights is None:
        weights = {
            "R": 1.0,
            "SC": 1.0,
            "EP": 1.0
        }

    df_pred, sim_df, p = predict_case1_rows(
        log_params=log_params,
        formulation_name=formulation_name,
        fit_data=fit_data
    )

    scales = get_case1_scales(
        formulation_name=formulation_name,
        fit_data=fit_data
    )

    residual_list = []

    n_receptor = max((df_pred["compartment"] == "R").sum(), 1)

    for _, row in df_pred.iterrows():
        compartment = row["compartment"]
        raw_error = row["model_ug_cm2"] - row["observed_ug_cm2"]
        scale = scales[compartment]

        if compartment == "R":
            weight_factor = np.sqrt(weights["R"] / n_receptor)
        else:
            weight_factor = np.sqrt(weights[compartment])

        residual = weight_factor * raw_error / scale
        residual_list.append(residual)

    return np.asarray(residual_list, dtype=float)


# -----------------------------
# Test residual function with initial parameters
# -----------------------------
initial_case1_params = {
    "Q0_eff_ug_cm2": 60.0,
    "D2_SC_cm2_h": 1e-6,
    "D3_EP_cm2_h": 1e-4,
    "P12_donor_SC": 1.0,
    "P34_EP_receptor": 1.0,
    "K34_cm_h": 0.1
}

initial_log_params = np.log([
    initial_case1_params["Q0_eff_ug_cm2"],
    initial_case1_params["D2_SC_cm2_h"],
    initial_case1_params["D3_EP_cm2_h"],
    initial_case1_params["P12_donor_SC"],
    initial_case1_params["P34_EP_receptor"],
    initial_case1_params["K34_cm_h"]
])

for formulation in ["Free KC", "KLZ-NPs"]:
    residual_vector = residuals_case1(
        log_params=initial_log_params,
        formulation_name=formulation,
        fit_data=fem_fit_data
    )

    df_pred_test, sim_df_test, unpacked_params = predict_case1_rows(
        log_params=initial_log_params,
        formulation_name=formulation,
        fit_data=fem_fit_data
    )

    print("\nFormulation:", formulation)
    print("Residual vector length:", len(residual_vector))
    print("Residual sum of squares:", np.sum(residual_vector**2))
    print("Unpacked test parameters:")
    print(unpacked_params)

    display(df_pred_test)


# -----------------------------
# Save a test prediction table for reproducibility
# -----------------------------
models_dir = f"{PROJECT_DIR}/outputs/models"

test_prediction_rows = []

for formulation in ["Free KC", "KLZ-NPs"]:
    df_pred_test, sim_df_test, unpacked_params = predict_case1_rows(
        log_params=initial_log_params,
        formulation_name=formulation,
        fit_data=fem_fit_data
    )

    test_prediction_rows.append(df_pred_test)

case1_test_predictions = pd.concat(test_prediction_rows, ignore_index=True)

case1_test_prediction_path = f"{models_dir}/case1_test_predictions_before_fitting.csv"
case1_test_predictions.to_csv(case1_test_prediction_path, index=False)

print("\nSaved Case 1 test prediction table:")
print(case1_test_prediction_path)

In [ ]:
# Cell 8: Case 1 fitting or archived-result loading

import numpy as np
import pandas as pd

if RUN_FULL_OPTIMIZATION:
    # Cell 8: Fit Case 1 four-layer FEM model

    import numpy as np
    import pandas as pd
    import os
    from scipy.optimize import least_squares

    # -----------------------------
    # Required variable check
    # -----------------------------
    required_names = [
        "residuals_case1",
        "predict_case1_rows",
        "simulate_case1_for_formulation",
        "fem_fit_data",
        "PROJECT_DIR"
    ]

    for name in required_names:
        if name not in globals():
            raise NameError(f"{name} is missing. Please run previous cells first.")


    # -----------------------------
    # Parameter bounds for Case 1
    # -----------------------------
    # These are broad but physically constrained bounds.
    # Parameters are fitted in log-space, so all parameters must be positive.

    case1_lower_bounds = {
        "Q0_eff_ug_cm2": 1.0,
        "D2_SC_cm2_h": 1e-10,
        "D3_EP_cm2_h": 1e-8,
        "P12_donor_SC": 1e-3,
        "P34_EP_receptor": 1e-3,
        "K34_cm_h": 1e-5
    }

    case1_upper_bounds = {
        "Q0_eff_ug_cm2": 5000.0,
        "D2_SC_cm2_h": 1e-2,
        "D3_EP_cm2_h": 1e0,
        "P12_donor_SC": 1e4,
        "P34_EP_receptor": 1e4,
        "K34_cm_h": 100.0
    }

    lower_vector = np.array([
        case1_lower_bounds[name] for name in CASE1_PARAMETER_NAMES
    ], dtype=float)

    upper_vector = np.array([
        case1_upper_bounds[name] for name in CASE1_PARAMETER_NAMES
    ], dtype=float)

    log_lower_vector = np.log(lower_vector)
    log_upper_vector = np.log(upper_vector)

    print("Case 1 parameter bounds")
    bounds_df = pd.DataFrame({
        "parameter": CASE1_PARAMETER_NAMES,
        "lower_bound": lower_vector,
        "upper_bound": upper_vector
    })
    display(bounds_df)


    # -----------------------------
    # Starting points
    # -----------------------------
    # Multiple starting points reduce the chance of getting trapped in one poor local minimum.

    starting_points = [
        [60.0, 1e-6, 1e-4, 1.0, 1.0, 0.1],
        [100.0, 1e-7, 1e-4, 0.1, 1.0, 0.5],
        [100.0, 1e-6, 1e-3, 1.0, 0.1, 1.0],
        [200.0, 1e-8, 1e-3, 10.0, 1.0, 1.0],
        [50.0, 1e-5, 1e-3, 0.1, 10.0, 0.1],
        [500.0, 1e-7, 1e-2, 10.0, 0.1, 5.0],
        [1000.0, 1e-6, 1e-2, 1.0, 1.0, 10.0],
        [100.0, 1e-9, 1e-5, 100.0, 10.0, 0.05]
    ]

    # Keep starting points inside bounds.
    clean_starting_points = []

    for start in starting_points:
        start_array = np.array(start, dtype=float)
        start_array = np.minimum(np.maximum(start_array, lower_vector * 1.01), upper_vector / 1.01)
        clean_starting_points.append(start_array)

    print("Number of starting points:", len(clean_starting_points))


    # -----------------------------
    # Fit function for one formulation
    # -----------------------------
    def fit_case1_formulation(formulation_name, weights=None, verbose=True):
        """
        Fit Case 1 model for one formulation using multiple starting points.
        """

        if weights is None:
            weights = {
                "R": 1.0,
                "SC": 1.0,
                "EP": 1.0
            }

        results = []

        for i, start in enumerate(clean_starting_points):
            x0 = np.log(start)

            result = least_squares(
                fun=lambda log_params: residuals_case1(
                    log_params=log_params,
                    formulation_name=formulation_name,
                    fit_data=fem_fit_data,
                    weights=weights
                ),
                x0=x0,
                bounds=(log_lower_vector, log_upper_vector),
                method="trf",
                loss="linear",
                max_nfev=500,
                xtol=1e-8,
                ftol=1e-8,
                gtol=1e-8
            )

            objective = np.sum(result.fun ** 2)

            results.append({
                "start_index": i,
                "result": result,
                "objective": objective,
                "success": result.success,
                "message": result.message
            })

            if verbose:
                print(
                    formulation_name,
                    "| start",
                    i,
                    "| success:",
                    result.success,
                    "| objective:",
                    objective
                )

        # Select best result based on objective value.
        best = min(results, key=lambda item: item["objective"])

        best_params = unpack_case1_log_params(best["result"].x)

        return best, best_params, results


    # -----------------------------
    # Fit both formulations
    # -----------------------------
    case1_fit_results = {}
    case1_all_start_results = []

    for formulation in ["Free KC", "KLZ-NPs"]:
        print("\nFitting Case 1:", formulation)

        best, best_params, all_results = fit_case1_formulation(
            formulation_name=formulation,
            verbose=True
        )

        case1_fit_results[formulation] = {
            "best": best,
            "best_params": best_params,
            "all_results": all_results
        }

        for item in all_results:
            case1_all_start_results.append({
                "formulation": formulation,
                "start_index": item["start_index"],
                "objective": item["objective"],
                "success": item["success"],
                "message": item["message"]
            })


    # -----------------------------
    # Create parameter table
    # -----------------------------
    case1_params_rows = []

    for formulation, fit_info in case1_fit_results.items():
        row = {
            "formulation": formulation,
            "model_case": "Case 1",
            "objective": fit_info["best"]["objective"],
            "success": fit_info["best"]["success"],
            "message": fit_info["best"]["message"]
        }

        row.update(fit_info["best_params"])
        case1_params_rows.append(row)

    case1_params_df = pd.DataFrame(case1_params_rows)

    print("\nBest Case 1 fitted parameters")
    display(case1_params_df)


    # -----------------------------
    # Generate fitted predictions
    # -----------------------------
    case1_prediction_rows = []
    case1_simulation_rows = []

    for formulation, fit_info in case1_fit_results.items():
        best_log_params = fit_info["best"]["result"].x

        df_pred, sim_df, p = predict_case1_rows(
            log_params=best_log_params,
            formulation_name=formulation,
            fit_data=fem_fit_data
        )

        df_pred["model_case"] = "Case 1"
        case1_prediction_rows.append(df_pred)

        # Simulate on a denser time grid for later plotting.
        dense_times = np.linspace(0, 24, 121)

        sim_dense_df, sol_dense, _ = simulate_case1_for_formulation(
            log_params=best_log_params,
            formulation_name=formulation,
            fit_data=pd.DataFrame({
                "formulation": [formulation] * len(dense_times),
                "time_h": dense_times,
                "compartment": ["R"] * len(dense_times),
                "observed_ug_cm2": [0.0] * len(dense_times),
                "data_type": ["dense_prediction"] * len(dense_times),
                "use_for_fitting": [False] * len(dense_times)
            })
        )

        sim_dense_df["formulation"] = formulation
        sim_dense_df["model_case"] = "Case 1"
        case1_simulation_rows.append(sim_dense_df)

    case1_predictions_df = pd.concat(case1_prediction_rows, ignore_index=True)
    case1_simulation_df = pd.concat(case1_simulation_rows, ignore_index=True)

    print("\nCase 1 fitted predictions")
    display(case1_predictions_df)


    # -----------------------------
    # Performance metrics
    # -----------------------------
    def compute_metrics(y_obs, y_pred):
        """
        Compute RMSE, MAE, and R2.
        """
        y_obs = np.asarray(y_obs, dtype=float)
        y_pred = np.asarray(y_pred, dtype=float)

        rmse = np.sqrt(np.mean((y_pred - y_obs) ** 2))
        mae = np.mean(np.abs(y_pred - y_obs))

        if len(y_obs) > 1 and np.sum((y_obs - np.mean(y_obs)) ** 2) > 0:
            r2 = 1.0 - np.sum((y_obs - y_pred) ** 2) / np.sum((y_obs - np.mean(y_obs)) ** 2)
        else:
            r2 = np.nan

        return rmse, mae, r2


    case1_metric_rows = []

    for formulation in ["Free KC", "KLZ-NPs"]:
        df_form = case1_predictions_df[
            case1_predictions_df["formulation"] == formulation
        ].copy()

        for target_name, compartment in [
            ("Receptor time-course", "R"),
            ("SC 24 h endpoint", "SC"),
            ("EP/dermis 24 h endpoint", "EP")
        ]:
            df_target = df_form[df_form["compartment"] == compartment].copy()

            rmse, mae, r2 = compute_metrics(
                y_obs=df_target["observed_ug_cm2"],
                y_pred=df_target["model_ug_cm2"]
            )

            case1_metric_rows.append({
                "formulation": formulation,
                "model_case": "Case 1",
                "target": target_name,
                "RMSE_ug_cm2": rmse,
                "MAE_ug_cm2": mae,
                "R2": r2,
                "n": len(df_target)
            })

    case1_metrics_df = pd.DataFrame(case1_metric_rows)

    print("\nCase 1 performance metrics")
    display(case1_metrics_df)


    # -----------------------------
    # Save outputs to the repository
    # -----------------------------
    models_dir = f"{PROJECT_DIR}/outputs/models"
    tables_dir = f"{PROJECT_DIR}/outputs/tables"

    case1_params_path = f"{models_dir}/case1_fitted_parameters.csv"
    case1_predictions_path = f"{models_dir}/case1_fitted_predictions.csv"
    case1_simulation_path = f"{models_dir}/case1_dense_simulation.csv"
    case1_metrics_path = f"{models_dir}/case1_performance_metrics.csv"
    case1_starts_path = f"{models_dir}/case1_all_start_results.csv"

    case1_params_df.to_csv(case1_params_path, index=False)
    case1_predictions_df.to_csv(case1_predictions_path, index=False)
    case1_simulation_df.to_csv(case1_simulation_path, index=False)
    case1_metrics_df.to_csv(case1_metrics_path, index=False)
    pd.DataFrame(case1_all_start_results).to_csv(case1_starts_path, index=False)

    print("\nSaved Case 1 outputs:")
    print(case1_params_path)
    print(case1_predictions_path)
    print(case1_simulation_path)
    print(case1_metrics_path)
    print(case1_starts_path)
else:
    print("Loading archived Case 1 fit outputs. Set RUN_FULL_OPTIMIZATION=True to refit.")
    models_dir = f"{PROJECT_DIR}/outputs/models"
    case1_params_df = pd.read_csv(f"{models_dir}/case1_fitted_parameters.csv")
    case1_predictions_df = pd.read_csv(f"{models_dir}/case1_fitted_predictions.csv")
    case1_simulation_df = pd.read_csv(f"{models_dir}/case1_dense_simulation.csv")
    case1_metrics_df = pd.read_csv(f"{models_dir}/case1_performance_metrics.csv")
    case1_all_start_results = pd.read_csv(f"{models_dir}/case1_all_start_results.csv").to_dict("records")

    def compute_metrics(y_obs, y_pred):
        y_obs = np.asarray(y_obs, dtype=float)
        y_pred = np.asarray(y_pred, dtype=float)
        rmse = np.sqrt(np.mean((y_pred - y_obs) ** 2))
        mae = np.mean(np.abs(y_pred - y_obs))
        if len(y_obs) > 1 and np.sum((y_obs - np.mean(y_obs)) ** 2) > 0:
            r2 = 1.0 - np.sum((y_obs - y_pred) ** 2) / np.sum((y_obs - np.mean(y_obs)) ** 2)
        else:
            r2 = np.nan
        return rmse, mae, r2

    display(case1_params_df)
    display(case1_metrics_df)


In [ ]:
# Cell 9.5: Restore Case 1 outputs and helper metric function

In [ ]:
# Cell 9: Case 1 diagnostic plots and parameter comparison

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Load Case 1 outputs if needed
# -----------------------------
models_dir = f"{PROJECT_DIR}/outputs/models"
figures_dir = f"{PROJECT_DIR}/outputs/figures"
tables_dir = f"{PROJECT_DIR}/outputs/tables"

if "case1_params_df" not in globals():
    case1_params_df = pd.read_csv(f"{models_dir}/case1_fitted_parameters.csv")

if "case1_predictions_df" not in globals():
    case1_predictions_df = pd.read_csv(f"{models_dir}/case1_fitted_predictions.csv")

if "case1_simulation_df" not in globals():
    case1_simulation_df = pd.read_csv(f"{models_dir}/case1_dense_simulation.csv")

if "case1_metrics_df" not in globals():
    case1_metrics_df = pd.read_csv(f"{models_dir}/case1_performance_metrics.csv")


# -----------------------------
# 1) Figure 3 diagnostic: receptor fit and 24 h endpoints
# -----------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Panel A: Free KC receptor fit
ax = axes[0, 0]

formulation = "Free KC"

df_obs = case1_predictions_df[
    (case1_predictions_df["formulation"] == formulation)
    & (case1_predictions_df["compartment"] == "R")
].sort_values("time_h")

df_sim = case1_simulation_df[
    case1_simulation_df["formulation"] == formulation
].sort_values("time_h")

ax.plot(
    df_sim["time_h"],
    df_sim["Q4_receptor_ug_cm2"],
    linewidth=2,
    label="Case 1 model"
)

ax.scatter(
    df_obs["time_h"],
    df_obs["observed_ug_cm2"],
    s=50,
    label="Experimental data"
)

ax.set_title("A. Free KC receptor fit")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


# Panel B: KLZ-NPs receptor fit
ax = axes[0, 1]

formulation = "KLZ-NPs"

df_obs = case1_predictions_df[
    (case1_predictions_df["formulation"] == formulation)
    & (case1_predictions_df["compartment"] == "R")
].sort_values("time_h")

df_sim = case1_simulation_df[
    case1_simulation_df["formulation"] == formulation
].sort_values("time_h")

ax.plot(
    df_sim["time_h"],
    df_sim["Q4_receptor_ug_cm2"],
    linewidth=2,
    label="Case 1 model"
)

ax.scatter(
    df_obs["time_h"],
    df_obs["observed_ug_cm2"],
    s=50,
    label="Experimental data"
)

ax.set_title("B. KLZ-NPs receptor fit")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


# Panel C: SC 24 h observed vs model
ax = axes[1, 0]

df_sc = case1_predictions_df[
    case1_predictions_df["compartment"] == "SC"
].copy()

x_positions = np.arange(len(df_sc))

ax.bar(
    x_positions - 0.18,
    df_sc["observed_ug_cm2"],
    width=0.36,
    label="Experimental"
)

ax.bar(
    x_positions + 0.18,
    df_sc["model_ug_cm2"],
    width=0.36,
    label="Case 1 model"
)

ax.set_xticks(x_positions)
ax.set_xticklabels(df_sc["formulation"], rotation=0)
ax.set_title("C. SC retention at 24 h")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


# Panel D: EP/dermis 24 h observed vs model
ax = axes[1, 1]

df_ep = case1_predictions_df[
    case1_predictions_df["compartment"] == "EP"
].copy()

x_positions = np.arange(len(df_ep))

ax.bar(
    x_positions - 0.18,
    df_ep["observed_ug_cm2"],
    width=0.36,
    label="Experimental"
)

ax.bar(
    x_positions + 0.18,
    df_ep["model_ug_cm2"],
    width=0.36,
    label="Case 1 model"
)

ax.set_xticks(x_positions)
ax.set_xticklabels(df_ep["formulation"], rotation=0)
ax.set_title("D. EP/dermis retention at 24 h")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.suptitle("Case 1 four-layer FEM fit to receptor permeation and 24 h skin-retention endpoints", y=1.02)
plt.tight_layout()

case1_fit_png = f"{figures_dir}/case1_figure3_fit_diagnostic.png"
case1_fit_pdf = f"{figures_dir}/case1_figure3_fit_diagnostic.pdf"
case1_fit_svg = f"{figures_dir}/case1_figure3_fit_diagnostic.svg"

plt.savefig(case1_fit_png, dpi=600, bbox_inches="tight")
plt.savefig(case1_fit_pdf, bbox_inches="tight")
plt.savefig(case1_fit_svg, bbox_inches="tight")

plt.show()

print("Saved Case 1 fit diagnostic figure:")
print(case1_fit_png)
print(case1_fit_pdf)
print(case1_fit_svg)


# -----------------------------
# 2) Residual plot
# -----------------------------
residual_df = case1_predictions_df.copy()

fig, ax = plt.subplots(figsize=(9, 5))

for formulation in ["Free KC", "KLZ-NPs"]:
    df_form = residual_df[
        (residual_df["formulation"] == formulation)
        & (residual_df["compartment"] == "R")
    ].sort_values("time_h")

    ax.plot(
        df_form["time_h"],
        df_form["error_ug_cm2"],
        marker="o",
        linewidth=2,
        label=formulation
    )

ax.axhline(0, linestyle="--", linewidth=1)

ax.set_title("Case 1 receptor residuals")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Model error (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

case1_resid_png = f"{figures_dir}/case1_receptor_residuals.png"
case1_resid_pdf = f"{figures_dir}/case1_receptor_residuals.pdf"
case1_resid_svg = f"{figures_dir}/case1_receptor_residuals.svg"

plt.savefig(case1_resid_png, dpi=600, bbox_inches="tight")
plt.savefig(case1_resid_pdf, bbox_inches="tight")
plt.savefig(case1_resid_svg, bbox_inches="tight")

plt.show()

print("Saved Case 1 residual figure:")
print(case1_resid_png)
print(case1_resid_pdf)
print(case1_resid_svg)


# -----------------------------
# 3) Parameter ratio table
# -----------------------------
free_row = case1_params_df[
    case1_params_df["formulation"] == "Free KC"
].iloc[0]

np_row = case1_params_df[
    case1_params_df["formulation"] == "KLZ-NPs"
].iloc[0]

parameter_columns = [
    "Q0_eff_ug_cm2",
    "D2_SC_cm2_h",
    "D3_EP_cm2_h",
    "P12_donor_SC",
    "P34_EP_receptor",
    "K34_cm_h"
]

ratio_rows = []

for param in parameter_columns:
    free_value = free_row[param]
    np_value = np_row[param]

    ratio_rows.append({
        "parameter": param,
        "Free_KC": free_value,
        "KLZ_NPs": np_value,
        "KLZ_NPs_to_Free_KC_ratio": np_value / free_value
    })

case1_parameter_ratio_df = pd.DataFrame(ratio_rows)

print("Case 1 parameter comparison")
display(case1_parameter_ratio_df)

case1_ratio_path = f"{tables_dir}/case1_parameter_ratio_table.csv"
case1_parameter_ratio_df.to_csv(case1_ratio_path, index=False)

print("Saved Case 1 parameter ratio table:")
print(case1_ratio_path)


# -----------------------------
# 4) Parameter ratio plot
# -----------------------------
fig, ax = plt.subplots(figsize=(10, 5))

x_positions = np.arange(len(case1_parameter_ratio_df))

ax.bar(
    x_positions,
    case1_parameter_ratio_df["KLZ_NPs_to_Free_KC_ratio"]
)

ax.axhline(1.0, linestyle="--", linewidth=1)

ax.set_xticks(x_positions)
ax.set_xticklabels(case1_parameter_ratio_df["parameter"], rotation=45, ha="right")
ax.set_ylabel("KLZ-NPs / Free KC ratio")
ax.set_title("Case 1 fitted parameter ratios")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

case1_ratio_png = f"{figures_dir}/case1_parameter_ratios.png"
case1_ratio_pdf = f"{figures_dir}/case1_parameter_ratios.pdf"
case1_ratio_svg = f"{figures_dir}/case1_parameter_ratios.svg"

plt.savefig(case1_ratio_png, dpi=600, bbox_inches="tight")
plt.savefig(case1_ratio_pdf, bbox_inches="tight")
plt.savefig(case1_ratio_svg, bbox_inches="tight")

plt.show()

print("Saved Case 1 parameter ratio figure:")
print(case1_ratio_png)
print(case1_ratio_pdf)
print(case1_ratio_svg)


# -----------------------------
# 5) Download outputs
# -----------------------------





In [ ]:
# Cell 10: Case 2 fitting or archived-result loading

import numpy as np
import pandas as pd

if RUN_FULL_OPTIMIZATION:
    # Cell 10: Fit Case 2 four-layer FEM model with fitted P23

    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.optimize import least_squares

    # -----------------------------
    # Required variable check
    # -----------------------------
    required_names = [
        "simulate_four_layer_model",
        "fem_fit_data",
        "get_case1_scales",
        "compute_metrics",
        "PROJECT_DIR"
    ]

    for name in required_names:
        if name not in globals():
            raise NameError(f"{name} is missing. Please run previous cells first.")

    models_dir = f"{PROJECT_DIR}/outputs/models"
    figures_dir = f"{PROJECT_DIR}/outputs/figures"
    tables_dir = f"{PROJECT_DIR}/outputs/tables"

    # -----------------------------
    # Case 2 parameter definition
    # -----------------------------
    # Case 2 fitted parameters:
    # Q0_eff, D2, D3, P12, P23, P34, K34
    #
    # Difference from Case 1:
    # P23 is fitted instead of fixed at 1.

    CASE2_PARAMETER_NAMES = [
        "Q0_eff_ug_cm2",
        "D2_SC_cm2_h",
        "D3_EP_cm2_h",
        "P12_donor_SC",
        "P23_SC_EP",
        "P34_EP_receptor",
        "K34_cm_h"
    ]

    print("Case 2 fitted parameters:")
    for name in CASE2_PARAMETER_NAMES:
        print("-", name)

    # -----------------------------
    # Convert log-parameters to real positive parameters
    # -----------------------------
    def unpack_case2_log_params(log_params):
        """
        Convert log-transformed Case 2 parameters into real positive parameters.
        """
        params = np.exp(np.asarray(log_params, dtype=float))

        return {
            "Q0_eff_ug_cm2": params[0],
            "D2_SC_cm2_h": params[1],
            "D3_EP_cm2_h": params[2],
            "P12_donor_SC": params[3],
            "P23_SC_EP": params[4],
            "P34_EP_receptor": params[5],
            "K34_cm_h": params[6]
        }

    # -----------------------------
    # Simulate Case 2 for one formulation
    # -----------------------------
    def simulate_case2_for_formulation(log_params, formulation_name, fit_data):
        """
        Run Case 2 model for one formulation and return model-predicted amounts.
        """
        p = unpack_case2_log_params(log_params)

        df_form = fit_data[
            fit_data["formulation"] == formulation_name
        ].copy()

        times = sorted(df_form["time_h"].unique())

        sim_df, sol = simulate_four_layer_model(
            times_h=times,
            Q0_eff_ug_cm2=p["Q0_eff_ug_cm2"],
            D2_cm2_h=p["D2_SC_cm2_h"],
            D3_cm2_h=p["D3_EP_cm2_h"],
            P12=p["P12_donor_SC"],
            P23=p["P23_SC_EP"],
            P34=p["P34_EP_receptor"],
            K34_cm_h=p["K34_cm_h"]
        )

        return sim_df, sol, p

    # -----------------------------
    # Predict Case 2 rows
    # -----------------------------
    def predict_case2_rows(log_params, formulation_name, fit_data):
        """
        Predict model values for all fitting rows of one formulation.
        """
        sim_df, sol, p = simulate_case2_for_formulation(
            log_params=log_params,
            formulation_name=formulation_name,
            fit_data=fit_data
        )

        sim_indexed = sim_df.set_index("time_h")

        df_form = fit_data[
            fit_data["formulation"] == formulation_name
        ].copy()

        predictions = []

        for _, row in df_form.iterrows():
            t = float(row["time_h"])
            compartment = row["compartment"]

            if compartment == "R":
                pred = sim_indexed.loc[t, "Q4_receptor_ug_cm2"]
            elif compartment == "SC":
                pred = sim_indexed.loc[t, "Q2_SC_ug_cm2"]
            elif compartment == "EP":
                pred = sim_indexed.loc[t, "Q3_EP_ug_cm2"]
            else:
                raise ValueError(f"Unknown compartment: {compartment}")

            predictions.append(pred)

        df_pred = df_form.copy()
        df_pred["model_ug_cm2"] = predictions
        df_pred["error_ug_cm2"] = df_pred["model_ug_cm2"] - df_pred["observed_ug_cm2"]

        return df_pred, sim_df, p

    # -----------------------------
    # Residual function for Case 2
    # -----------------------------
    def residuals_case2(log_params, formulation_name, fit_data, weights=None):
        """
        Compute normalized residual vector for Case 2.
        """
        if weights is None:
            weights = {
                "R": 1.0,
                "SC": 1.0,
                "EP": 1.0
            }

        df_pred, sim_df, p = predict_case2_rows(
            log_params=log_params,
            formulation_name=formulation_name,
            fit_data=fit_data
        )

        scales = get_case1_scales(
            formulation_name=formulation_name,
            fit_data=fit_data
        )

        residual_list = []

        n_receptor = max((df_pred["compartment"] == "R").sum(), 1)

        for _, row in df_pred.iterrows():
            compartment = row["compartment"]
            raw_error = row["model_ug_cm2"] - row["observed_ug_cm2"]
            scale = scales[compartment]

            if compartment == "R":
                weight_factor = np.sqrt(weights["R"] / n_receptor)
            else:
                weight_factor = np.sqrt(weights[compartment])

            residual = weight_factor * raw_error / scale
            residual_list.append(residual)

        return np.asarray(residual_list, dtype=float)

    # -----------------------------
    # Parameter bounds for Case 2
    # -----------------------------
    case2_lower_bounds = {
        "Q0_eff_ug_cm2": 1.0,
        "D2_SC_cm2_h": 1e-10,
        "D3_EP_cm2_h": 1e-8,
        "P12_donor_SC": 1e-3,
        "P23_SC_EP": 1e-3,
        "P34_EP_receptor": 1e-3,
        "K34_cm_h": 1e-5
    }

    case2_upper_bounds = {
        "Q0_eff_ug_cm2": 5000.0,
        "D2_SC_cm2_h": 1e-2,
        "D3_EP_cm2_h": 1e0,
        "P12_donor_SC": 1e4,
        "P23_SC_EP": 1e4,
        "P34_EP_receptor": 1e4,
        "K34_cm_h": 100.0
    }

    case2_lower_vector = np.array([
        case2_lower_bounds[name] for name in CASE2_PARAMETER_NAMES
    ], dtype=float)

    case2_upper_vector = np.array([
        case2_upper_bounds[name] for name in CASE2_PARAMETER_NAMES
    ], dtype=float)

    case2_log_lower_vector = np.log(case2_lower_vector)
    case2_log_upper_vector = np.log(case2_upper_vector)

    case2_bounds_df = pd.DataFrame({
        "parameter": CASE2_PARAMETER_NAMES,
        "lower_bound": case2_lower_vector,
        "upper_bound": case2_upper_vector
    })

    print("Case 2 parameter bounds")
    display(case2_bounds_df)

    case2_bounds_path = f"{tables_dir}/case2_parameter_bounds.csv"
    case2_bounds_df.to_csv(case2_bounds_path, index=False)

    # -----------------------------
    # Starting points
    # -----------------------------
    # Use Case 1 fitted parameters as strong starting points, plus variations in P23.

    starting_points_case2 = []

    if "case1_params_df" in globals():
        for _, row in case1_params_df.iterrows():
            base = [
                row["Q0_eff_ug_cm2"],
                row["D2_SC_cm2_h"],
                row["D3_EP_cm2_h"],
                row["P12_donor_SC"],
                1.0,
                row["P34_EP_receptor"],
                row["K34_cm_h"]
            ]

            for p23 in [0.05, 0.1, 0.5, 1.0, 2.0, 10.0, 50.0]:
                temp = base.copy()
                temp[4] = p23
                starting_points_case2.append(temp)

    # Additional generic starting points
    starting_points_case2 += [
        [100.0, 1e-6, 1e-4, 1.0, 1.0, 1.0, 0.1],
        [200.0, 1e-7, 1e-3, 10.0, 0.1, 1.0, 1.0],
        [500.0, 1e-6, 1e-3, 1.0, 10.0, 0.1, 5.0],
        [1000.0, 1e-8, 1e-2, 100.0, 1.0, 10.0, 0.5],
        [1000.0, 1e-5, 1e-3, 0.1, 0.1, 10.0, 10.0]
    ]

    clean_starting_points_case2 = []

    for start in starting_points_case2:
        start_array = np.array(start, dtype=float)
        start_array = np.minimum(
            np.maximum(start_array, case2_lower_vector * 1.01),
            case2_upper_vector / 1.01
        )
        clean_starting_points_case2.append(start_array)

    print("Number of Case 2 starting points:", len(clean_starting_points_case2))

    # -----------------------------
    # Fit function for one formulation
    # -----------------------------
    def fit_case2_formulation(formulation_name, weights=None, verbose=True):
        """
        Fit Case 2 model for one formulation using multiple starting points.
        """
        if weights is None:
            weights = {
                "R": 1.0,
                "SC": 1.0,
                "EP": 1.0
            }

        results = []

        for i, start in enumerate(clean_starting_points_case2):
            x0 = np.log(start)

            result = least_squares(
                fun=lambda log_params: residuals_case2(
                    log_params=log_params,
                    formulation_name=formulation_name,
                    fit_data=fem_fit_data,
                    weights=weights
                ),
                x0=x0,
                bounds=(case2_log_lower_vector, case2_log_upper_vector),
                method="trf",
                loss="linear",
                max_nfev=800,
                xtol=1e-8,
                ftol=1e-8,
                gtol=1e-8
            )

            objective = np.sum(result.fun ** 2)

            results.append({
                "start_index": i,
                "result": result,
                "objective": objective,
                "success": result.success,
                "message": result.message
            })

            if verbose:
                print(
                    formulation_name,
                    "| start",
                    i,
                    "| success:",
                    result.success,
                    "| objective:",
                    objective
                )

        best = min(results, key=lambda item: item["objective"])
        best_params = unpack_case2_log_params(best["result"].x)

        return best, best_params, results

    # -----------------------------
    # Fit both formulations
    # -----------------------------
    case2_fit_results = {}
    case2_all_start_results = []

    for formulation in ["Free KC", "KLZ-NPs"]:
        print("\nFitting Case 2:", formulation)

        best, best_params, all_results = fit_case2_formulation(
            formulation_name=formulation,
            verbose=True
        )

        case2_fit_results[formulation] = {
            "best": best,
            "best_params": best_params,
            "all_results": all_results
        }

        for item in all_results:
            case2_all_start_results.append({
                "formulation": formulation,
                "start_index": item["start_index"],
                "objective": item["objective"],
                "success": item["success"],
                "message": item["message"]
            })

    # -----------------------------
    # Create parameter table
    # -----------------------------
    case2_params_rows = []

    for formulation, fit_info in case2_fit_results.items():
        row = {
            "formulation": formulation,
            "model_case": "Case 2",
            "objective": fit_info["best"]["objective"],
            "success": fit_info["best"]["success"],
            "message": fit_info["best"]["message"]
        }

        row.update(fit_info["best_params"])
        case2_params_rows.append(row)

    case2_params_df = pd.DataFrame(case2_params_rows)

    print("\nBest Case 2 fitted parameters")
    display(case2_params_df)

    # -----------------------------
    # Generate fitted predictions and dense simulation
    # -----------------------------
    case2_prediction_rows = []
    case2_simulation_rows = []

    for formulation, fit_info in case2_fit_results.items():
        best_log_params = fit_info["best"]["result"].x

        df_pred, sim_df, p = predict_case2_rows(
            log_params=best_log_params,
            formulation_name=formulation,
            fit_data=fem_fit_data
        )

        df_pred["model_case"] = "Case 2"
        case2_prediction_rows.append(df_pred)

        dense_times = np.linspace(0, 24, 121)

        sim_dense_df, sol_dense, _ = simulate_case2_for_formulation(
            log_params=best_log_params,
            formulation_name=formulation,
            fit_data=pd.DataFrame({
                "formulation": [formulation] * len(dense_times),
                "time_h": dense_times,
                "compartment": ["R"] * len(dense_times),
                "observed_ug_cm2": [0.0] * len(dense_times),
                "data_type": ["dense_prediction"] * len(dense_times),
                "use_for_fitting": [False] * len(dense_times)
            })
        )

        sim_dense_df["formulation"] = formulation
        sim_dense_df["model_case"] = "Case 2"
        case2_simulation_rows.append(sim_dense_df)

    case2_predictions_df = pd.concat(case2_prediction_rows, ignore_index=True)
    case2_simulation_df = pd.concat(case2_simulation_rows, ignore_index=True)

    print("\nCase 2 fitted predictions")
    display(case2_predictions_df)

    # -----------------------------
    # Performance metrics
    # -----------------------------
    case2_metric_rows = []

    for formulation in ["Free KC", "KLZ-NPs"]:
        df_form = case2_predictions_df[
            case2_predictions_df["formulation"] == formulation
        ].copy()

        for target_name, compartment in [
            ("Receptor time-course", "R"),
            ("SC 24 h endpoint", "SC"),
            ("EP/dermis 24 h endpoint", "EP")
        ]:
            df_target = df_form[df_form["compartment"] == compartment].copy()

            rmse, mae, r2 = compute_metrics(
                y_obs=df_target["observed_ug_cm2"],
                y_pred=df_target["model_ug_cm2"]
            )

            case2_metric_rows.append({
                "formulation": formulation,
                "model_case": "Case 2",
                "target": target_name,
                "RMSE_ug_cm2": rmse,
                "MAE_ug_cm2": mae,
                "R2": r2,
                "n": len(df_target)
            })

    case2_metrics_df = pd.DataFrame(case2_metric_rows)

    print("\nCase 2 performance metrics")
    display(case2_metrics_df)

    # -----------------------------
    # Save outputs
    # -----------------------------
    case2_params_path = f"{models_dir}/case2_fitted_parameters.csv"
    case2_predictions_path = f"{models_dir}/case2_fitted_predictions.csv"
    case2_simulation_path = f"{models_dir}/case2_dense_simulation.csv"
    case2_metrics_path = f"{models_dir}/case2_performance_metrics.csv"
    case2_starts_path = f"{models_dir}/case2_all_start_results.csv"

    case2_params_df.to_csv(case2_params_path, index=False)
    case2_predictions_df.to_csv(case2_predictions_path, index=False)
    case2_simulation_df.to_csv(case2_simulation_path, index=False)
    case2_metrics_df.to_csv(case2_metrics_path, index=False)
    pd.DataFrame(case2_all_start_results).to_csv(case2_starts_path, index=False)

    print("\nSaved Case 2 outputs:")
    print(case2_params_path)
    print(case2_predictions_path)
    print(case2_simulation_path)
    print(case2_metrics_path)
    print(case2_starts_path)
    print(case2_bounds_path)

    # -----------------------------
    # Optional downloads
    # -----------------------------
else:
    print("Loading archived Case 2 fit outputs. Set RUN_FULL_OPTIMIZATION=True to refit.")
    models_dir = f"{PROJECT_DIR}/outputs/models"
    tables_dir = f"{PROJECT_DIR}/outputs/tables"
    case2_params_df = pd.read_csv(f"{models_dir}/case2_fitted_parameters.csv")
    case2_predictions_df = pd.read_csv(f"{models_dir}/case2_fitted_predictions.csv")
    case2_simulation_df = pd.read_csv(f"{models_dir}/case2_dense_simulation.csv")
    case2_metrics_df = pd.read_csv(f"{models_dir}/case2_performance_metrics.csv")
    case2_all_start_results = pd.read_csv(f"{models_dir}/case2_all_start_results.csv").to_dict("records")
    case2_bounds_path = f"{tables_dir}/case2_parameter_bounds.csv"
    display(case2_params_df)
    display(case2_metrics_df)


In [ ]:
# Cell 10.5: Rebuild Case 2 predictions, metrics, and diagnostic plots

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Required variable check
# -----------------------------
required_names = [
    "simulate_four_layer_model",
    "fem_fit_data",
    "PROJECT_DIR"
]

for name in required_names:
    if name not in globals():
        raise NameError(f"{name} is missing. Please run Cells 4, 5, and 6 first.")

models_dir = f"{PROJECT_DIR}/outputs/models"
figures_dir = f"{PROJECT_DIR}/outputs/figures"
tables_dir = f"{PROJECT_DIR}/outputs/tables"

# -----------------------------
# Restore Case 2 parameter table if needed
# -----------------------------
if "case2_params_df" not in globals():
    case2_params_df = pd.DataFrame([
        {
            "formulation": "Free KC",
            "model_case": "Case 2",
            "objective": 0.001194,
            "success": True,
            "message": "`xtol` termination condition is satisfied.",
            "Q0_eff_ug_cm2": 485.968206,
            "D2_SC_cm2_h": 0.000004,
            "D3_EP_cm2_h": 0.000810,
            "P12_donor_SC": 1.313211,
            "P23_SC_EP": 1.000158,
            "P34_EP_receptor": 0.002985,
            "K34_cm_h": 0.840574
        },
        {
            "formulation": "KLZ-NPs",
            "model_case": "Case 2",
            "objective": 0.000059,
            "success": True,
            "message": "`xtol` termination condition is satisfied.",
            "Q0_eff_ug_cm2": 863.822949,
            "D2_SC_cm2_h": 0.000004,
            "D3_EP_cm2_h": 0.000670,
            "P12_donor_SC": 1.025471,
            "P23_SC_EP": 4.425177,
            "P34_EP_receptor": 0.023180,
            "K34_cm_h": 3.036170
        }
    ])

print("Case 2 parameter table:")
display(case2_params_df)

# -----------------------------
# Metric helper
# -----------------------------
def compute_metrics(y_obs, y_pred):
    """
    Compute RMSE, MAE, and R2.
    """
    y_obs = np.asarray(y_obs, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    rmse = np.sqrt(np.mean((y_pred - y_obs) ** 2))
    mae = np.mean(np.abs(y_pred - y_obs))

    if len(y_obs) > 1 and np.sum((y_obs - np.mean(y_obs)) ** 2) > 0:
        r2 = 1.0 - np.sum((y_obs - y_pred) ** 2) / np.sum((y_obs - np.mean(y_obs)) ** 2)
    else:
        r2 = np.nan

    return rmse, mae, r2

# -----------------------------
# Simulation helper from Case 2 parameter row
# -----------------------------
def simulate_case2_from_row(param_row, times_h):
    """
    Run Case 2 simulation using one row of fitted parameters.
    """
    sim_df, sol = simulate_four_layer_model(
        times_h=times_h,
        Q0_eff_ug_cm2=float(param_row["Q0_eff_ug_cm2"]),
        D2_cm2_h=float(param_row["D2_SC_cm2_h"]),
        D3_cm2_h=float(param_row["D3_EP_cm2_h"]),
        P12=float(param_row["P12_donor_SC"]),
        P23=float(param_row["P23_SC_EP"]),
        P34=float(param_row["P34_EP_receptor"]),
        K34_cm_h=float(param_row["K34_cm_h"])
    )

    return sim_df, sol

# -----------------------------
# Build Case 2 predictions
# -----------------------------
case2_prediction_rows = []
case2_simulation_rows = []

for _, param_row in case2_params_df.iterrows():
    formulation = param_row["formulation"]

    df_form = fem_fit_data[
        fem_fit_data["formulation"] == formulation
    ].copy()

    fit_times = sorted(df_form["time_h"].unique())

    sim_fit_df, sol_fit = simulate_case2_from_row(
        param_row=param_row,
        times_h=fit_times
    )

    sim_indexed = sim_fit_df.set_index("time_h")

    predictions = []

    for _, obs_row in df_form.iterrows():
        t = float(obs_row["time_h"])
        compartment = obs_row["compartment"]

        if compartment == "R":
            pred = sim_indexed.loc[t, "Q4_receptor_ug_cm2"]
        elif compartment == "SC":
            pred = sim_indexed.loc[t, "Q2_SC_ug_cm2"]
        elif compartment == "EP":
            pred = sim_indexed.loc[t, "Q3_EP_ug_cm2"]
        else:
            raise ValueError(f"Unknown compartment: {compartment}")

        predictions.append(pred)

    df_pred = df_form.copy()
    df_pred["model_ug_cm2"] = predictions
    df_pred["error_ug_cm2"] = df_pred["model_ug_cm2"] - df_pred["observed_ug_cm2"]
    df_pred["model_case"] = "Case 2"

    case2_prediction_rows.append(df_pred)

    dense_times = np.linspace(0, 24, 121)

    sim_dense_df, sol_dense = simulate_case2_from_row(
        param_row=param_row,
        times_h=dense_times
    )

    sim_dense_df["formulation"] = formulation
    sim_dense_df["model_case"] = "Case 2"

    case2_simulation_rows.append(sim_dense_df)

case2_predictions_df = pd.concat(case2_prediction_rows, ignore_index=True)
case2_simulation_df = pd.concat(case2_simulation_rows, ignore_index=True)

print("Case 2 fitted predictions:")
display(case2_predictions_df)

# -----------------------------
# Case 2 metrics
# -----------------------------
case2_metric_rows = []

for formulation in ["Free KC", "KLZ-NPs"]:
    df_form = case2_predictions_df[
        case2_predictions_df["formulation"] == formulation
    ].copy()

    for target_name, compartment in [
        ("Receptor time-course", "R"),
        ("SC 24 h endpoint", "SC"),
        ("EP/dermis 24 h endpoint", "EP")
    ]:
        df_target = df_form[
            df_form["compartment"] == compartment
        ].copy()

        rmse, mae, r2 = compute_metrics(
            y_obs=df_target["observed_ug_cm2"],
            y_pred=df_target["model_ug_cm2"]
        )

        case2_metric_rows.append({
            "formulation": formulation,
            "model_case": "Case 2",
            "target": target_name,
            "RMSE_ug_cm2": rmse,
            "MAE_ug_cm2": mae,
            "R2": r2,
            "n": len(df_target)
        })

case2_metrics_df = pd.DataFrame(case2_metric_rows)

print("Case 2 performance metrics:")
display(case2_metrics_df)

# -----------------------------
# Save Case 2 outputs
# -----------------------------
case2_params_path = f"{models_dir}/case2_fitted_parameters.csv"
case2_predictions_path = f"{models_dir}/case2_fitted_predictions.csv"
case2_simulation_path = f"{models_dir}/case2_dense_simulation.csv"
case2_metrics_path = f"{models_dir}/case2_performance_metrics.csv"

case2_params_df.to_csv(case2_params_path, index=False)
case2_predictions_df.to_csv(case2_predictions_path, index=False)
case2_simulation_df.to_csv(case2_simulation_path, index=False)
case2_metrics_df.to_csv(case2_metrics_path, index=False)

print("Saved Case 2 outputs:")
print(case2_params_path)
print(case2_predictions_path)
print(case2_simulation_path)
print(case2_metrics_path)

# -----------------------------
# Figure: Case 2 fit diagnostic
# -----------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Panel A: Free KC receptor fit
ax = axes[0, 0]
formulation = "Free KC"

df_obs = case2_predictions_df[
    (case2_predictions_df["formulation"] == formulation)
    & (case2_predictions_df["compartment"] == "R")
].sort_values("time_h")

df_sim = case2_simulation_df[
    case2_simulation_df["formulation"] == formulation
].sort_values("time_h")

ax.plot(df_sim["time_h"], df_sim["Q4_receptor_ug_cm2"], linewidth=2, label="Case 2 model")
ax.scatter(df_obs["time_h"], df_obs["observed_ug_cm2"], s=50, label="Experimental data")
ax.set_title("A. Free KC receptor fit")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Panel B: KLZ-NPs receptor fit
ax = axes[0, 1]
formulation = "KLZ-NPs"

df_obs = case2_predictions_df[
    (case2_predictions_df["formulation"] == formulation)
    & (case2_predictions_df["compartment"] == "R")
].sort_values("time_h")

df_sim = case2_simulation_df[
    case2_simulation_df["formulation"] == formulation
].sort_values("time_h")

ax.plot(df_sim["time_h"], df_sim["Q4_receptor_ug_cm2"], linewidth=2, label="Case 2 model")
ax.scatter(df_obs["time_h"], df_obs["observed_ug_cm2"], s=50, label="Experimental data")
ax.set_title("B. KLZ-NPs receptor fit")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Panel C: SC endpoint
ax = axes[1, 0]

df_sc = case2_predictions_df[
    case2_predictions_df["compartment"] == "SC"
].copy()

x_positions = np.arange(len(df_sc))

ax.bar(x_positions - 0.18, df_sc["observed_ug_cm2"], width=0.36, label="Experimental")
ax.bar(x_positions + 0.18, df_sc["model_ug_cm2"], width=0.36, label="Case 2 model")
ax.set_xticks(x_positions)
ax.set_xticklabels(df_sc["formulation"])
ax.set_title("C. SC retention at 24 h")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Panel D: EP endpoint
ax = axes[1, 1]

df_ep = case2_predictions_df[
    case2_predictions_df["compartment"] == "EP"
].copy()

x_positions = np.arange(len(df_ep))

ax.bar(x_positions - 0.18, df_ep["observed_ug_cm2"], width=0.36, label="Experimental")
ax.bar(x_positions + 0.18, df_ep["model_ug_cm2"], width=0.36, label="Case 2 model")
ax.set_xticks(x_positions)
ax.set_xticklabels(df_ep["formulation"])
ax.set_title("D. EP/dermis retention at 24 h")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.suptitle("Case 2 four-layer FEM fit to receptor permeation and 24 h skin-retention endpoints", y=1.02)
plt.tight_layout()

case2_fit_png = f"{figures_dir}/case2_figure_fit_diagnostic.png"
case2_fit_pdf = f"{figures_dir}/case2_figure_fit_diagnostic.pdf"
case2_fit_svg = f"{figures_dir}/case2_figure_fit_diagnostic.svg"

plt.savefig(case2_fit_png, dpi=600, bbox_inches="tight")
plt.savefig(case2_fit_pdf, bbox_inches="tight")
plt.savefig(case2_fit_svg, bbox_inches="tight")

plt.show()

print("Saved Case 2 fit diagnostic figure:")
print(case2_fit_png)
print(case2_fit_pdf)
print(case2_fit_svg)

# -----------------------------
# Figure: Case 2 receptor residuals
# -----------------------------
fig, ax = plt.subplots(figsize=(9, 5))

for formulation in ["Free KC", "KLZ-NPs"]:
    df_form = case2_predictions_df[
        (case2_predictions_df["formulation"] == formulation)
        & (case2_predictions_df["compartment"] == "R")
    ].sort_values("time_h")

    ax.plot(
        df_form["time_h"],
        df_form["error_ug_cm2"],
        marker="o",
        linewidth=2,
        label=formulation
    )

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title("Case 2 receptor residuals")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Model error (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

case2_resid_png = f"{figures_dir}/case2_receptor_residuals.png"
case2_resid_pdf = f"{figures_dir}/case2_receptor_residuals.pdf"
case2_resid_svg = f"{figures_dir}/case2_receptor_residuals.svg"

plt.savefig(case2_resid_png, dpi=600, bbox_inches="tight")
plt.savefig(case2_resid_pdf, bbox_inches="tight")
plt.savefig(case2_resid_svg, bbox_inches="tight")

plt.show()

print("Saved Case 2 residual figure:")
print(case2_resid_png)
print(case2_resid_pdf)
print(case2_resid_svg)

# -----------------------------
# Download key outputs
# -----------------------------




In [ ]:
# Cell 11: Compare Case 1 and Case 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Define project folders
# -----------------------------
models_dir = f"{PROJECT_DIR}/outputs/models"
figures_dir = f"{PROJECT_DIR}/outputs/figures"
tables_dir = f"{PROJECT_DIR}/outputs/tables"

# -----------------------------
# Load saved outputs
# -----------------------------
case1_params_df = pd.read_csv(f"{models_dir}/case1_fitted_parameters.csv")
case1_predictions_df = pd.read_csv(f"{models_dir}/case1_fitted_predictions.csv")
case1_metrics_df = pd.read_csv(f"{models_dir}/case1_performance_metrics.csv")
case1_simulation_df = pd.read_csv(f"{models_dir}/case1_dense_simulation.csv")

case2_params_df = pd.read_csv(f"{models_dir}/case2_fitted_parameters.csv")
case2_predictions_df = pd.read_csv(f"{models_dir}/case2_fitted_predictions.csv")
case2_metrics_df = pd.read_csv(f"{models_dir}/case2_performance_metrics.csv")
case2_simulation_df = pd.read_csv(f"{models_dir}/case2_dense_simulation.csv")

print("Loaded Case 1 and Case 2 outputs.")

# -----------------------------
# 1) Metrics comparison table
# -----------------------------
metrics_comparison_df = pd.concat(
    [case1_metrics_df, case2_metrics_df],
    ignore_index=True
)

metrics_comparison_df = metrics_comparison_df.sort_values(
    ["formulation", "target", "model_case"]
).reset_index(drop=True)

print("Performance metrics comparison:")
display(metrics_comparison_df)

metrics_comparison_path = f"{tables_dir}/case1_case2_performance_comparison.csv"
metrics_comparison_df.to_csv(metrics_comparison_path, index=False)

# -----------------------------
# 2) Objective and parameter count comparison
# -----------------------------
case1_objective_df = case1_params_df[["formulation", "model_case", "objective"]].copy()
case1_objective_df["n_fitted_parameters"] = 6
case1_objective_df["n_fitting_points"] = 7

case2_objective_df = case2_params_df[["formulation", "model_case", "objective"]].copy()
case2_objective_df["n_fitted_parameters"] = 7
case2_objective_df["n_fitting_points"] = 7

objective_comparison_df = pd.concat(
    [case1_objective_df, case2_objective_df],
    ignore_index=True
).sort_values(["formulation", "model_case"]).reset_index(drop=True)

# Approximate AIC and BIC using normalized objective as RSS-like value.
# This is only a relative diagnostic because the residuals are normalized.
objective_comparison_df["AIC_approx"] = (
    objective_comparison_df["n_fitting_points"]
    * np.log(objective_comparison_df["objective"] / objective_comparison_df["n_fitting_points"])
    + 2 * objective_comparison_df["n_fitted_parameters"]
)

objective_comparison_df["BIC_approx"] = (
    objective_comparison_df["n_fitting_points"]
    * np.log(objective_comparison_df["objective"] / objective_comparison_df["n_fitting_points"])
    + objective_comparison_df["n_fitted_parameters"]
    * np.log(objective_comparison_df["n_fitting_points"])
)

print("Objective comparison:")
display(objective_comparison_df)

objective_comparison_path = f"{tables_dir}/case1_case2_objective_comparison.csv"
objective_comparison_df.to_csv(objective_comparison_path, index=False)

# -----------------------------
# 3) Receptor metric pivot table
# -----------------------------
receptor_metrics_df = metrics_comparison_df[
    metrics_comparison_df["target"] == "Receptor time-course"
].copy()

receptor_metric_pivot = receptor_metrics_df.pivot_table(
    index="formulation",
    columns="model_case",
    values=["RMSE_ug_cm2", "MAE_ug_cm2", "R2"]
)

print("Receptor metric pivot:")
display(receptor_metric_pivot)

receptor_metric_path = f"{tables_dir}/case1_case2_receptor_metric_pivot.csv"
receptor_metric_pivot.to_csv(receptor_metric_path)

# -----------------------------
# 4) Parameter comparison table
# -----------------------------
case1_params_for_merge = case1_params_df.copy()
case2_params_for_merge = case2_params_df.copy()

case1_params_for_merge["P23_SC_EP"] = 1.0

parameter_columns = [
    "Q0_eff_ug_cm2",
    "D2_SC_cm2_h",
    "D3_EP_cm2_h",
    "P12_donor_SC",
    "P23_SC_EP",
    "P34_EP_receptor",
    "K34_cm_h"
]

parameter_comparison_rows = []

for formulation in ["Free KC", "KLZ-NPs"]:
    c1 = case1_params_for_merge[
        case1_params_for_merge["formulation"] == formulation
    ].iloc[0]

    c2 = case2_params_for_merge[
        case2_params_for_merge["formulation"] == formulation
    ].iloc[0]

    for param in parameter_columns:
        parameter_comparison_rows.append({
            "formulation": formulation,
            "parameter": param,
            "Case_1": c1[param],
            "Case_2": c2[param],
            "Case_2_to_Case_1_ratio": c2[param] / c1[param]
        })

parameter_comparison_df = pd.DataFrame(parameter_comparison_rows)

print("Parameter comparison:")
display(parameter_comparison_df)

parameter_comparison_path = f"{tables_dir}/case1_case2_parameter_comparison.csv"
parameter_comparison_df.to_csv(parameter_comparison_path, index=False)

# -----------------------------
# 5) Model selection summary
# -----------------------------
selection_rows = []

for formulation in ["Free KC", "KLZ-NPs"]:
    c1_obj = objective_comparison_df[
        (objective_comparison_df["formulation"] == formulation)
        & (objective_comparison_df["model_case"] == "Case 1")
    ]["objective"].iloc[0]

    c2_obj = objective_comparison_df[
        (objective_comparison_df["formulation"] == formulation)
        & (objective_comparison_df["model_case"] == "Case 2")
    ]["objective"].iloc[0]

    c1_rmse = receptor_metrics_df[
        (receptor_metrics_df["formulation"] == formulation)
        & (receptor_metrics_df["model_case"] == "Case 1")
    ]["RMSE_ug_cm2"].iloc[0]

    c2_rmse = receptor_metrics_df[
        (receptor_metrics_df["formulation"] == formulation)
        & (receptor_metrics_df["model_case"] == "Case 2")
    ]["RMSE_ug_cm2"].iloc[0]

    p23_case2 = case2_params_df[
        case2_params_df["formulation"] == formulation
    ]["P23_SC_EP"].iloc[0]

    objective_improvement_percent = 100 * (c1_obj - c2_obj) / c1_obj
    rmse_improvement_percent = 100 * (c1_rmse - c2_rmse) / c1_rmse

    if formulation == "Free KC":
        recommended_role = "Use Case 1 as the main conservative model"
        interpretation = "Case 2 adds little value because P23 remains approximately 1."
    else:
        recommended_role = "Use Case 2 as an extended nanoparticle-sensitive model"
        interpretation = "Case 2 improves KLZ-NPs fit and suggests altered SC-to-EP partitioning."

    selection_rows.append({
        "formulation": formulation,
        "Case_1_objective": c1_obj,
        "Case_2_objective": c2_obj,
        "objective_improvement_percent": objective_improvement_percent,
        "Case_1_receptor_RMSE": c1_rmse,
        "Case_2_receptor_RMSE": c2_rmse,
        "RMSE_improvement_percent": rmse_improvement_percent,
        "Case_2_P23_SC_EP": p23_case2,
        "recommended_role": recommended_role,
        "interpretation": interpretation
    })

model_selection_summary_df = pd.DataFrame(selection_rows)

print("Model selection summary:")
display(model_selection_summary_df)

model_selection_path = f"{tables_dir}/case1_case2_model_selection_summary.csv"
model_selection_summary_df.to_csv(model_selection_path, index=False)

# -----------------------------
# 6) Plot receptor fits for Case 1 and Case 2
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, formulation in zip(axes, ["Free KC", "KLZ-NPs"]):
    obs = case1_predictions_df[
        (case1_predictions_df["formulation"] == formulation)
        & (case1_predictions_df["compartment"] == "R")
    ].sort_values("time_h")

    c1_sim = case1_simulation_df[
        case1_simulation_df["formulation"] == formulation
    ].sort_values("time_h")

    c2_sim = case2_simulation_df[
        case2_simulation_df["formulation"] == formulation
    ].sort_values("time_h")

    ax.plot(
        c1_sim["time_h"],
        c1_sim["Q4_receptor_ug_cm2"],
        linewidth=2,
        label="Case 1"
    )

    ax.plot(
        c2_sim["time_h"],
        c2_sim["Q4_receptor_ug_cm2"],
        linewidth=2,
        linestyle="--",
        label="Case 2"
    )

    ax.scatter(
        obs["time_h"],
        obs["observed_ug_cm2"],
        s=50,
        label="Experimental"
    )

    ax.set_title(f"{formulation} receptor fit comparison")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Ketoconazole amount (µg/cm²)")
    ax.legend(frameon=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()

fit_comparison_png = f"{figures_dir}/case1_case2_receptor_fit_comparison.png"
fit_comparison_pdf = f"{figures_dir}/case1_case2_receptor_fit_comparison.pdf"
fit_comparison_svg = f"{figures_dir}/case1_case2_receptor_fit_comparison.svg"

plt.savefig(fit_comparison_png, dpi=600, bbox_inches="tight")
plt.savefig(fit_comparison_pdf, bbox_inches="tight")
plt.savefig(fit_comparison_svg, bbox_inches="tight")

plt.show()

print("Saved receptor fit comparison figure:")
print(fit_comparison_png)
print(fit_comparison_pdf)
print(fit_comparison_svg)

# -----------------------------
# 7) Plot receptor RMSE comparison
# -----------------------------
rmse_plot_df = receptor_metrics_df.copy()

fig, ax = plt.subplots(figsize=(7, 4.5))

x_labels = ["Free KC", "KLZ-NPs"]
x = np.arange(len(x_labels))
width = 0.35

case1_rmse = [
    rmse_plot_df[
        (rmse_plot_df["formulation"] == label)
        & (rmse_plot_df["model_case"] == "Case 1")
    ]["RMSE_ug_cm2"].iloc[0]
    for label in x_labels
]

case2_rmse = [
    rmse_plot_df[
        (rmse_plot_df["formulation"] == label)
        & (rmse_plot_df["model_case"] == "Case 2")
    ]["RMSE_ug_cm2"].iloc[0]
    for label in x_labels
]

ax.bar(x - width / 2, case1_rmse, width=width, label="Case 1")
ax.bar(x + width / 2, case2_rmse, width=width, label="Case 2")

ax.set_xticks(x)
ax.set_xticklabels(x_labels)
ax.set_ylabel("Receptor RMSE (µg/cm²)")
ax.set_title("Receptor RMSE comparison")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

rmse_comparison_png = f"{figures_dir}/case1_case2_receptor_rmse_comparison.png"
rmse_comparison_pdf = f"{figures_dir}/case1_case2_receptor_rmse_comparison.pdf"
rmse_comparison_svg = f"{figures_dir}/case1_case2_receptor_rmse_comparison.svg"

plt.savefig(rmse_comparison_png, dpi=600, bbox_inches="tight")
plt.savefig(rmse_comparison_pdf, bbox_inches="tight")
plt.savefig(rmse_comparison_svg, bbox_inches="tight")

plt.show()

print("Saved receptor RMSE comparison figure:")
print(rmse_comparison_png)
print(rmse_comparison_pdf)
print(rmse_comparison_svg)

# -----------------------------
# 8) Download outputs
# -----------------------------




In [ ]:
# Cell 12: Build selected final model tables

import os
import numpy as np
import pandas as pd

# -----------------------------
# Define project folders
# -----------------------------
models_dir = f"{PROJECT_DIR}/outputs/models"
tables_dir = f"{PROJECT_DIR}/outputs/tables"

# -----------------------------
# Load saved Case 1 and Case 2 outputs
# -----------------------------
case1_params_df = pd.read_csv(f"{models_dir}/case1_fitted_parameters.csv")
case1_predictions_df = pd.read_csv(f"{models_dir}/case1_fitted_predictions.csv")
case1_metrics_df = pd.read_csv(f"{models_dir}/case1_performance_metrics.csv")
case1_simulation_df = pd.read_csv(f"{models_dir}/case1_dense_simulation.csv")

case2_params_df = pd.read_csv(f"{models_dir}/case2_fitted_parameters.csv")
case2_predictions_df = pd.read_csv(f"{models_dir}/case2_fitted_predictions.csv")
case2_metrics_df = pd.read_csv(f"{models_dir}/case2_performance_metrics.csv")
case2_simulation_df = pd.read_csv(f"{models_dir}/case2_dense_simulation.csv")

# -----------------------------
# Select final model by formulation
# -----------------------------
selected_model_rules = {
    "Free KC": "Case 1",
    "KLZ-NPs": "Case 2"
}

selected_params_rows = []
selected_predictions_rows = []
selected_metrics_rows = []
selected_simulation_rows = []

for formulation, selected_case in selected_model_rules.items():

    if selected_case == "Case 1":
        params_source = case1_params_df
        predictions_source = case1_predictions_df
        metrics_source = case1_metrics_df
        simulation_source = case1_simulation_df
    elif selected_case == "Case 2":
        params_source = case2_params_df
        predictions_source = case2_predictions_df
        metrics_source = case2_metrics_df
        simulation_source = case2_simulation_df
    else:
        raise ValueError(f"Unknown selected case: {selected_case}")

    selected_params_rows.append(
        params_source[
            params_source["formulation"] == formulation
        ].copy()
    )

    selected_predictions_rows.append(
        predictions_source[
            predictions_source["formulation"] == formulation
        ].copy()
    )

    selected_metrics_rows.append(
        metrics_source[
            metrics_source["formulation"] == formulation
        ].copy()
    )

    selected_simulation_rows.append(
        simulation_source[
            simulation_source["formulation"] == formulation
        ].copy()
    )

selected_params_df = pd.concat(selected_params_rows, ignore_index=True)
selected_predictions_df = pd.concat(selected_predictions_rows, ignore_index=True)
selected_metrics_df = pd.concat(selected_metrics_rows, ignore_index=True)
selected_simulation_df = pd.concat(selected_simulation_rows, ignore_index=True)

# -----------------------------
# Add selection rationale
# -----------------------------
selection_rationale = pd.DataFrame([
    {
        "formulation": "Free KC",
        "selected_model": "Case 1",
        "rationale": "Case 2 did not materially improve the receptor fit, and P23 remained approximately 1."
    },
    {
        "formulation": "KLZ-NPs",
        "selected_model": "Case 2",
        "rationale": "Case 2 substantially reduced receptor RMSE and estimated a formulation-sensitive P23 value."
    }
])

# -----------------------------
# Save selected final model outputs
# -----------------------------
selected_params_path = f"{models_dir}/selected_final_model_parameters.csv"
selected_predictions_path = f"{models_dir}/selected_final_model_predictions.csv"
selected_metrics_path = f"{models_dir}/selected_final_model_metrics.csv"
selected_simulation_path = f"{models_dir}/selected_final_model_dense_simulation.csv"
selection_rationale_path = f"{tables_dir}/selected_final_model_rationale.csv"

selected_params_df.to_csv(selected_params_path, index=False)
selected_predictions_df.to_csv(selected_predictions_path, index=False)
selected_metrics_df.to_csv(selected_metrics_path, index=False)
selected_simulation_df.to_csv(selected_simulation_path, index=False)
selection_rationale.to_csv(selection_rationale_path, index=False)

# -----------------------------
# Display final selected outputs
# -----------------------------
print("Selected final model parameters:")
display(selected_params_df)

print("Selected final model performance metrics:")
display(selected_metrics_df)

print("Selected final model rationale:")
display(selection_rationale)

print("Saved selected final model files:")
print(selected_params_path)
print(selected_predictions_path)
print(selected_metrics_path)
print(selected_simulation_path)
print(selection_rationale_path)

In [ ]:
# Cell 13: Generate final model-predicted concentration profiles across skin layers

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Required variable check
# -----------------------------
required_names = [
    "simulate_four_layer_model",
    "mesh_nodes_df",
    "layer_geometry",
    "PROJECT_DIR"
]

for name in required_names:
    if name not in globals():
        raise NameError(f"{name} is missing. Please run Cells 4, 5, and 6 first.")

# -----------------------------
# Project folders
# -----------------------------
models_dir = f"{PROJECT_DIR}/outputs/models"
figures_dir = f"{PROJECT_DIR}/outputs/figures"
tables_dir = f"{PROJECT_DIR}/outputs/tables"

# -----------------------------
# Load selected final model parameters
# -----------------------------
selected_params_df = pd.read_csv(
    f"{models_dir}/selected_final_model_parameters.csv"
)

# In Case 1, P23 was fixed at 1.0.
# The saved table may show NaN for P23, so we replace it with 1.0.
selected_params_df["P23_SC_EP"] = selected_params_df["P23_SC_EP"].fillna(1.0)

fixed_selected_params_path = f"{models_dir}/selected_final_model_parameters_fixed_P23.csv"
selected_params_df.to_csv(fixed_selected_params_path, index=False)

print("Selected final model parameters with P23 fixed where needed:")
display(selected_params_df)

# -----------------------------
# Times for spatial concentration profiles
# -----------------------------
profile_times_h = np.array([0, 1, 4, 8, 12, 24], dtype=float)
plot_times_h = [1, 4, 8, 12, 24]

# -----------------------------
# Helper function to simulate from selected parameter row
# -----------------------------
def simulate_selected_model_from_row(param_row, times_h):
    """
    Run the selected final model for one formulation using its final parameters.
    """
    sim_df, sol = simulate_four_layer_model(
        times_h=times_h,
        Q0_eff_ug_cm2=float(param_row["Q0_eff_ug_cm2"]),
        D2_cm2_h=float(param_row["D2_SC_cm2_h"]),
        D3_cm2_h=float(param_row["D3_EP_cm2_h"]),
        P12=float(param_row["P12_donor_SC"]),
        P23=float(param_row["P23_SC_EP"]),
        P34=float(param_row["P34_EP_receptor"]),
        K34_cm_h=float(param_row["K34_cm_h"])
    )

    return sim_df, sol

# -----------------------------
# Build concentration profile table
# -----------------------------
profile_rows = []
amount_rows = []

for _, param_row in selected_params_df.iterrows():
    formulation = param_row["formulation"]
    model_case = param_row["model_case"]

    sim_amounts_df, sol = simulate_selected_model_from_row(
        param_row=param_row,
        times_h=profile_times_h
    )

    sim_amounts_df["formulation"] = formulation
    sim_amounts_df["model_case"] = model_case
    amount_rows.append(sim_amounts_df)

    for time_index, time_h in enumerate(sol.t):
        C_t = sol.y[:, time_index]

        for _, node in mesh_nodes_df.iterrows():
            profile_rows.append({
                "formulation": formulation,
                "model_case": model_case,
                "time_h": float(time_h),
                "global_node_id": int(node["global_node_id"]),
                "layer_id": int(node["layer_id"]),
                "layer_name": node["layer_name"],
                "x_global_cm": float(node["x_global_cm"]),
                "concentration_ug_cm3": float(C_t[int(node["global_node_id"])])
            })

concentration_profiles_df = pd.DataFrame(profile_rows)
spatial_amounts_df = pd.concat(amount_rows, ignore_index=True)

# -----------------------------
# Add skin-domain coordinate
# -----------------------------
skin_start_cm = layer_geometry.loc[
    layer_geometry["short_name"] == "SC",
    "start_cm"
].iloc[0]

sc_end_cm = layer_geometry.loc[
    layer_geometry["short_name"] == "SC",
    "end_cm"
].iloc[0]

skin_end_cm = layer_geometry.loc[
    layer_geometry["short_name"] == "EP/dermis",
    "end_cm"
].iloc[0]

concentration_profiles_df["x_skin_cm"] = (
    concentration_profiles_df["x_global_cm"] - skin_start_cm
)

concentration_profiles_df["x_skin_um"] = (
    concentration_profiles_df["x_skin_cm"] * 10000.0
)

sc_ep_interface_um = (sc_end_cm - skin_start_cm) * 10000.0
skin_end_um = (skin_end_cm - skin_start_cm) * 10000.0

# -----------------------------
# Save profile data
# -----------------------------
profile_data_path = f"{models_dir}/selected_final_model_spatial_concentration_profiles.csv"
amounts_data_path = f"{models_dir}/selected_final_model_spatial_profile_amounts.csv"

concentration_profiles_df.to_csv(profile_data_path, index=False)
spatial_amounts_df.to_csv(amounts_data_path, index=False)

print("Saved spatial concentration profile data:")
print(profile_data_path)
print(amounts_data_path)

# -----------------------------
# Plot Figure 4: skin-domain concentration profiles
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, formulation in zip(axes, ["Free KC", "KLZ-NPs"]):

    df_form = concentration_profiles_df[
        (concentration_profiles_df["formulation"] == formulation)
        & (concentration_profiles_df["layer_id"].isin([2, 3]))
        & (concentration_profiles_df["time_h"].isin(plot_times_h))
    ].copy()

    for time_h in plot_times_h:
        df_time = df_form[
            df_form["time_h"] == time_h
        ].sort_values(["x_skin_um", "layer_id", "global_node_id"])

        ax.plot(
            df_time["x_skin_um"],
            df_time["concentration_ug_cm3"],
            linewidth=2,
            label=f"{time_h:g} h"
        )

    # SC/EP interface
    ax.axvline(
        sc_ep_interface_um,
        linestyle="--",
        linewidth=1
    )

    ax.text(
        sc_ep_interface_um / 2,
        ax.get_ylim()[1] * 0.92,
        "SC",
        ha="center",
        va="top"
    )

    ax.text(
        (sc_ep_interface_um + skin_end_um) / 2,
        ax.get_ylim()[1] * 0.92,
        "EP/dermis",
        ha="center",
        va="top"
    )

    ax.set_title(f"{formulation} skin concentration profiles")
    ax.set_xlabel("Position across skin domain (µm)")
    ax.set_ylabel("Ketoconazole concentration (µg/cm³)")
    ax.legend(frameon=False, title="Time")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle(
    "Model-predicted ketoconazole concentration profiles across SC and EP/dermis",
    y=1.03
)

plt.tight_layout()

figure4_png = f"{figures_dir}/figure4_selected_model_skin_concentration_profiles.png"
figure4_pdf = f"{figures_dir}/figure4_selected_model_skin_concentration_profiles.pdf"
figure4_svg = f"{figures_dir}/figure4_selected_model_skin_concentration_profiles.svg"

plt.savefig(figure4_png, dpi=600, bbox_inches="tight")
plt.savefig(figure4_pdf, bbox_inches="tight")
plt.savefig(figure4_svg, bbox_inches="tight")

plt.show()

print("Saved Figure 4 skin concentration profiles:")
print(figure4_png)
print(figure4_pdf)
print(figure4_svg)

# -----------------------------
# Plot supplementary full computational-domain profiles
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, formulation in zip(axes, ["Free KC", "KLZ-NPs"]):

    df_form = concentration_profiles_df[
        (concentration_profiles_df["formulation"] == formulation)
        & (concentration_profiles_df["time_h"].isin(plot_times_h))
    ].copy()

    for time_h in plot_times_h:
        df_time = df_form[
            df_form["time_h"] == time_h
        ].sort_values(["x_global_cm", "layer_id", "global_node_id"])

        ax.plot(
            df_time["x_global_cm"],
            df_time["concentration_ug_cm3"],
            linewidth=2,
            label=f"{time_h:g} h"
        )

    # Layer boundaries
    for _, layer in layer_geometry.iterrows():
        ax.axvline(
            layer["start_cm"],
            linestyle="--",
            linewidth=0.8
        )

    ax.axvline(
        layer_geometry["end_cm"].iloc[-1],
        linestyle="--",
        linewidth=0.8
    )

    ax.set_title(f"{formulation} full computational domain")
    ax.set_xlabel("Position across computational domain (cm)")
    ax.set_ylabel("Ketoconazole concentration (µg/cm³)")
    ax.legend(frameon=False, title="Time")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle(
    "Supplementary model-predicted concentration profiles across four computational compartments",
    y=1.03
)

plt.tight_layout()

supp_full_png = f"{figures_dir}/supplementary_full_computational_domain_concentration_profiles.png"
supp_full_pdf = f"{figures_dir}/supplementary_full_computational_domain_concentration_profiles.pdf"
supp_full_svg = f"{figures_dir}/supplementary_full_computational_domain_concentration_profiles.svg"

plt.savefig(supp_full_png, dpi=600, bbox_inches="tight")
plt.savefig(supp_full_pdf, bbox_inches="tight")
plt.savefig(supp_full_svg, bbox_inches="tight")

plt.show()

print("Saved supplementary full-domain profiles:")
print(supp_full_png)
print(supp_full_pdf)
print(supp_full_svg)

# -----------------------------
# Download outputs
# -----------------------------




In [ ]:
# Cell 14: Publication-ready split skin concentration profiles

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Project folders
# -----------------------------
models_dir = f"{PROJECT_DIR}/outputs/models"
figures_dir = f"{PROJECT_DIR}/outputs/figures"

# -----------------------------
# Load spatial concentration profiles
# -----------------------------
profile_data_path = f"{models_dir}/selected_final_model_spatial_concentration_profiles.csv"
concentration_profiles_df = pd.read_csv(profile_data_path)

# -----------------------------
# Plot settings
# -----------------------------
plot_times_h = [1, 4, 8, 12, 24]

# Layer IDs:
# 2 = SC
# 3 = EP/dermis

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

panel_info = [
    ("Free KC", 2, "A. Free KC: SC"),
    ("Free KC", 3, "B. Free KC: EP/dermis"),
    ("KLZ-NPs", 2, "C. KLZ-NPs: SC"),
    ("KLZ-NPs", 3, "D. KLZ-NPs: EP/dermis")
]

for ax, (formulation, layer_id, title) in zip(axes.flatten(), panel_info):

    df_layer = concentration_profiles_df[
        (concentration_profiles_df["formulation"] == formulation)
        & (concentration_profiles_df["layer_id"] == layer_id)
        & (concentration_profiles_df["time_h"].isin(plot_times_h))
    ].copy()

    # For each layer, reset x-axis to local position within that layer.
    layer_start_um = df_layer["x_skin_um"].min()
    df_layer["x_layer_um"] = df_layer["x_skin_um"] - layer_start_um

    for time_h in plot_times_h:
        df_time = df_layer[
            df_layer["time_h"] == time_h
        ].sort_values("x_layer_um")

        ax.plot(
            df_time["x_layer_um"],
            df_time["concentration_ug_cm3"],
            linewidth=2,
            label=f"{time_h:g} h"
        )

    if layer_id == 2:
        ax.set_xlabel("Position within SC (µm)")
    else:
        ax.set_xlabel("Position within EP/dermis (µm)")

    ax.set_ylabel("Ketoconazole concentration (µg/cm³)")
    ax.set_title(title)
    ax.legend(frameon=False, title="Time")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle(
    "Model-predicted ketoconazole concentration profiles within individual skin layers",
    y=1.02
)

plt.tight_layout()

figure4_split_png = f"{figures_dir}/figure4_publication_split_layer_concentration_profiles.png"
figure4_split_pdf = f"{figures_dir}/figure4_publication_split_layer_concentration_profiles.pdf"
figure4_split_svg = f"{figures_dir}/figure4_publication_split_layer_concentration_profiles.svg"

plt.savefig(figure4_split_png, dpi=600, bbox_inches="tight")
plt.savefig(figure4_split_pdf, bbox_inches="tight")
plt.savefig(figure4_split_svg, bbox_inches="tight")

plt.show()

print("Saved publication-ready split-layer Figure 4:")
print(figure4_split_png)
print(figure4_split_pdf)
print(figure4_split_svg)



In [ ]:
# Cell 15: Journal-ready Figure 4 with shared legend

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Project folders
# -----------------------------
models_dir = f"{PROJECT_DIR}/outputs/models"
figures_dir = f"{PROJECT_DIR}/outputs/figures"

# -----------------------------
# Load spatial concentration profiles
# -----------------------------
profile_data_path = f"{models_dir}/selected_final_model_spatial_concentration_profiles.csv"
concentration_profiles_df = pd.read_csv(profile_data_path)

# -----------------------------
# Plot settings
# -----------------------------
plot_times_h = [1, 4, 8, 12, 24]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

panel_info = [
    ("Free KC", 2, "A. Free KC: SC"),
    ("Free KC", 3, "B. Free KC: EP/dermis"),
    ("KLZ-NPs", 2, "C. KLZ-NPs: SC"),
    ("KLZ-NPs", 3, "D. KLZ-NPs: EP/dermis")
]

handles_for_legend = None
labels_for_legend = None

for ax, (formulation, layer_id, title) in zip(axes.flatten(), panel_info):

    df_layer = concentration_profiles_df[
        (concentration_profiles_df["formulation"] == formulation)
        & (concentration_profiles_df["layer_id"] == layer_id)
        & (concentration_profiles_df["time_h"].isin(plot_times_h))
    ].copy()

    layer_start_um = df_layer["x_skin_um"].min()
    df_layer["x_layer_um"] = df_layer["x_skin_um"] - layer_start_um

    for time_h in plot_times_h:
        df_time = df_layer[
            df_layer["time_h"] == time_h
        ].sort_values("x_layer_um")

        line, = ax.plot(
            df_time["x_layer_um"],
            df_time["concentration_ug_cm3"],
            linewidth=2,
            label=f"{time_h:g} h"
        )

    if handles_for_legend is None:
        handles_for_legend, labels_for_legend = ax.get_legend_handles_labels()

    if layer_id == 2:
        ax.set_xlabel("Position within SC (µm)")
    else:
        ax.set_xlabel("Position within EP/dermis (µm)")

    ax.set_ylabel("Ketoconazole concentration (µg/cm³)")
    ax.set_title(title)

    # Remove individual legends
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# -----------------------------
# Shared legend
# -----------------------------
fig.legend(
    handles_for_legend,
    labels_for_legend,
    title="Time",
    loc="lower center",
    ncol=5,
    frameon=False,
    bbox_to_anchor=(0.5, -0.02)
)

fig.suptitle(
    "Model-predicted skin-layer concentration profiles",
    y=0.98
)

plt.tight_layout(rect=[0, 0.06, 1, 0.95])

# -----------------------------
# Save outputs
# -----------------------------
figure4_journal_png = f"{figures_dir}/figure4_journal_ready_skin_layer_concentration_profiles.png"
figure4_journal_pdf = f"{figures_dir}/figure4_journal_ready_skin_layer_concentration_profiles.pdf"
figure4_journal_svg = f"{figures_dir}/figure4_journal_ready_skin_layer_concentration_profiles.svg"

plt.savefig(figure4_journal_png, dpi=600, bbox_inches="tight")
plt.savefig(figure4_journal_pdf, bbox_inches="tight")
plt.savefig(figure4_journal_svg, bbox_inches="tight")

plt.show()

print("Saved journal-ready Figure 4:")
print(figure4_journal_png)
print(figure4_journal_pdf)
print(figure4_journal_svg)



In [ ]:
# Cell 15.2a: Use repository-relative project path for a fresh notebook session

import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Mount the repository if needed

# Use repository-relative project directory

models_dir = f"{PROJECT_DIR}/outputs/models"
figures_dir = f"{PROJECT_DIR}/outputs/figures"
tables_dir = f"{PROJECT_DIR}/outputs/tables"

print("PROJECT_DIR restored:")
print(PROJECT_DIR)

print("\nFolder check:")
print("models exists:", os.path.exists(models_dir))
print("figures exists:", os.path.exists(figures_dir))
print("tables exists:", os.path.exists(tables_dir))

# Check key files needed for receptor concentration analysis
key_files = [
    f"{models_dir}/selected_final_model_dense_simulation.csv",
    f"{models_dir}/selected_final_model_parameters.csv"
]

for file_path in key_files:
    print(os.path.basename(file_path), "exists:", os.path.exists(file_path))

# Search for receptor_time_data.csv
receptor_candidates = glob.glob(
    f"{PROJECT_DIR}/**/receptor_time_data.csv",
    recursive=True
)

print("\nFound receptor_time_data.csv files:")
for path in receptor_candidates:
    print(path)

if len(receptor_candidates) > 0:
    receptor_df = pd.read_csv(receptor_candidates[0])
    print("\nLoaded receptor_df from:")
    print(receptor_candidates[0])
    display(receptor_df)
else:
    print("\nNo receptor_time_data.csv found. If needed, rerun the data setup cell.")

In [ ]:
# Cell 15.2: Experimental and model receptor concentration-time profiles

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Project folders
# -----------------------------
models_dir = f"{PROJECT_DIR}/outputs/models"
figures_dir = f"{PROJECT_DIR}/outputs/figures"
tables_dir = f"{PROJECT_DIR}/outputs/tables"

# -----------------------------
# Experimental constants
# -----------------------------
# Article reports Franz cell diffusion area = 2.2 cm^2
# and receptor volume = 6.5 mL.

AREA_CM2 = 2.2
RECEPTOR_VOLUME_ML = 6.5

RECEPTOR_EQUIVALENT_THICKNESS_CM = RECEPTOR_VOLUME_ML / AREA_CM2

print("Receptor equivalent thickness, V_R/A (cm):", RECEPTOR_EQUIVALENT_THICKNESS_CM)

# -----------------------------
# Load experimental receptor amount data
# -----------------------------
# receptor_df should already exist from earlier cells.
# If not, try to load it from saved project files.

if "receptor_df" not in globals():
    possible_receptor_paths = [
        f"{PROJECT_DIR}/data/processed/receptor_time_data.csv",
        f"{PROJECT_DIR}/data/raw/receptor_time_data.csv",
        f"{PROJECT_DIR}/receptor_time_data.csv"
    ]

    receptor_df = None

    for path in possible_receptor_paths:
        if os.path.exists(path):
            receptor_df = pd.read_csv(path)
            print("Loaded receptor_df from:", path)
            break

    if receptor_df is None:
        raise FileNotFoundError(
            "Could not find receptor_time_data.csv. Please run the data setup cell again."
        )

print("Experimental receptor amount data:")
display(receptor_df)

# -----------------------------
# Standardize receptor dataframe
# -----------------------------
# Expected columns:
# formulation, time_h, Q_receptor_ug_cm2

required_columns = ["formulation", "time_h", "Q_receptor_ug_cm2"]

for col in required_columns:
    if col not in receptor_df.columns:
        raise KeyError(
            f"Missing column: {col}. Available columns are: {receptor_df.columns.tolist()}"
        )

experimental_receptor_conc_df = receptor_df.copy()

# Convert amount per area to receptor concentration.
# C_R = Q_R * A / V_R
# Since 1 mL = 1 cm^3, unit becomes ug/mL, numerically same as ug/cm^3.

experimental_receptor_conc_df["C_receptor_exp_ug_mL"] = (
    experimental_receptor_conc_df["Q_receptor_ug_cm2"]
    * AREA_CM2
    / RECEPTOR_VOLUME_ML
)

experimental_receptor_conc_df["C_receptor_exp_ug_cm3"] = (
    experimental_receptor_conc_df["C_receptor_exp_ug_mL"]
)

print("Experimental receptor concentration data:")
display(experimental_receptor_conc_df)

# -----------------------------
# Load selected final model simulation
# -----------------------------
selected_simulation_path = f"{models_dir}/selected_final_model_dense_simulation.csv"

if "selected_simulation_df" not in globals():
    selected_simulation_df = pd.read_csv(selected_simulation_path)

print("Selected final model dense simulation:")
display(selected_simulation_df.head())

# -----------------------------
# Convert model receptor amount to receptor concentration
# -----------------------------
model_receptor_conc_df = selected_simulation_df.copy()

if "Q4_receptor_ug_cm2" not in model_receptor_conc_df.columns:
    raise KeyError(
        "Q4_receptor_ug_cm2 is missing from selected_simulation_df."
    )

model_receptor_conc_df["C_receptor_model_ug_mL"] = (
    model_receptor_conc_df["Q4_receptor_ug_cm2"]
    * AREA_CM2
    / RECEPTOR_VOLUME_ML
)

model_receptor_conc_df["C_receptor_model_ug_cm3"] = (
    model_receptor_conc_df["C_receptor_model_ug_mL"]
)

# -----------------------------
# Model predictions at experimental time points
# -----------------------------
prediction_rows = []

for _, row in experimental_receptor_conc_df.iterrows():
    formulation = row["formulation"]
    time_h = float(row["time_h"])

    df_model_form = model_receptor_conc_df[
        model_receptor_conc_df["formulation"] == formulation
    ].copy()

    # Dense simulation includes regular times.
    # Use interpolation to get model value exactly at experimental time.
    c_model = np.interp(
        time_h,
        df_model_form["time_h"],
        df_model_form["C_receptor_model_ug_mL"]
    )

    q_model = np.interp(
        time_h,
        df_model_form["time_h"],
        df_model_form["Q4_receptor_ug_cm2"]
    )

    prediction_rows.append({
        "formulation": formulation,
        "time_h": time_h,
        "Q_receptor_exp_ug_cm2": row["Q_receptor_ug_cm2"],
        "C_receptor_exp_ug_mL": row["C_receptor_exp_ug_mL"],
        "Q_receptor_model_ug_cm2": q_model,
        "C_receptor_model_ug_mL": c_model,
        "C_error_ug_mL": c_model - row["C_receptor_exp_ug_mL"]
    })

receptor_concentration_comparison_df = pd.DataFrame(prediction_rows)

print("Experimental vs model receptor concentration comparison:")
display(receptor_concentration_comparison_df)

# -----------------------------
# Save receptor concentration tables
# -----------------------------
experimental_receptor_conc_path = f"{tables_dir}/experimental_receptor_concentration.csv"
model_receptor_conc_path = f"{models_dir}/selected_final_model_receptor_concentration_dense.csv"
receptor_conc_comparison_path = f"{tables_dir}/experimental_vs_model_receptor_concentration_comparison.csv"

experimental_receptor_conc_df.to_csv(experimental_receptor_conc_path, index=False)
model_receptor_conc_df.to_csv(model_receptor_conc_path, index=False)
receptor_concentration_comparison_df.to_csv(receptor_conc_comparison_path, index=False)

print("Saved receptor concentration tables:")
print(experimental_receptor_conc_path)
print(model_receptor_conc_path)
print(receptor_conc_comparison_path)

# -----------------------------
# Plot receptor concentration-time profiles
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)

for ax, formulation in zip(axes, ["Free KC", "KLZ-NPs"]):

    df_exp = experimental_receptor_conc_df[
        experimental_receptor_conc_df["formulation"] == formulation
    ].sort_values("time_h")

    df_model = model_receptor_conc_df[
        model_receptor_conc_df["formulation"] == formulation
    ].sort_values("time_h")

    selected_case = df_model["model_case"].iloc[0] if "model_case" in df_model.columns else "Selected model"

    ax.plot(
        df_model["time_h"],
        df_model["C_receptor_model_ug_mL"],
        linewidth=2,
        label=f"Model ({selected_case})"
    )

    ax.scatter(
        df_exp["time_h"],
        df_exp["C_receptor_exp_ug_mL"],
        s=55,
        label="Experimental"
    )

    ax.set_title(f"{formulation} receptor concentration")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Receptor concentration (µg/mL)")
    ax.legend(frameon=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle(
    "Experimental and model-predicted receptor concentration-time profiles",
    y=1.03
)

plt.tight_layout()

receptor_conc_png = f"{figures_dir}/receptor_concentration_time_profiles.png"
receptor_conc_pdf = f"{figures_dir}/receptor_concentration_time_profiles.pdf"
receptor_conc_svg = f"{figures_dir}/receptor_concentration_time_profiles.svg"

plt.savefig(receptor_conc_png, dpi=600, bbox_inches="tight")
plt.savefig(receptor_conc_pdf, bbox_inches="tight")
plt.savefig(receptor_conc_svg, bbox_inches="tight")

plt.show()

print("Saved receptor concentration-time profile figure:")
print(receptor_conc_png)
print(receptor_conc_pdf)
print(receptor_conc_svg)

# -----------------------------
# Download outputs
# -----------------------------



In [ ]:
# Cell 16: Mesh independence test for the selected final four-layer FEM model

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp

# -----------------------------
# Use repository-relative project path
# -----------------------------


models_dir = f"{PROJECT_DIR}/outputs/models"
figures_dir = f"{PROJECT_DIR}/outputs/figures"
tables_dir = f"{PROJECT_DIR}/outputs/tables"

os.makedirs(models_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)
os.makedirs(tables_dir, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)

# -----------------------------
# Experimental and model constants
# -----------------------------
AREA_CM2 = 2.2
DONOR_VOLUME_ML = 1.0
RECEPTOR_VOLUME_ML = 6.5

SKIN_THICKNESS_CM = 0.09
SC_THICKNESS_CM = 0.002
EP_THICKNESS_CM = SKIN_THICKNESS_CM - SC_THICKNESS_CM

L1_DONOR_CM = DONOR_VOLUME_ML / AREA_CM2
L2_SC_CM = SC_THICKNESS_CM
L3_EP_CM = EP_THICKNESS_CM
L4_RECEPTOR_CM = RECEPTOR_VOLUME_ML / AREA_CM2

D1_FIXED_CM2_H = 10.0
D4_FIXED_CM2_H = 10.0

K12_FIXED_CM_H = 100.0
K23_FIXED_CM_H = 100.0

print("Layer lengths:")
print("Donor equivalent thickness (cm):", L1_DONOR_CM)
print("SC thickness (cm):", L2_SC_CM)
print("EP/dermis thickness (cm):", L3_EP_CM)
print("Receptor equivalent thickness (cm):", L4_RECEPTOR_CM)

# -----------------------------
# Load selected final model parameters
# -----------------------------
selected_params_path = f"{models_dir}/selected_final_model_parameters.csv"

if not os.path.exists(selected_params_path):
    raise FileNotFoundError(
        f"Missing selected final model parameter file: {selected_params_path}"
    )

selected_params_df = pd.read_csv(selected_params_path)

# Case 1 has P23 fixed at 1.0.
selected_params_df["P23_SC_EP"] = selected_params_df["P23_SC_EP"].fillna(1.0)

print("Selected final model parameters used for mesh test:")
display(selected_params_df)

# -----------------------------
# Mesh definitions
# -----------------------------
mesh_cases = {
    "Coarse": {
        "n_donor": 10,
        "n_SC": 5,
        "n_EP": 50,
        "n_receptor": 15
    },
    "Base": {
        "n_donor": 20,
        "n_SC": 10,
        "n_EP": 100,
        "n_receptor": 30
    },
    "Fine": {
        "n_donor": 40,
        "n_SC": 20,
        "n_EP": 200,
        "n_receptor": 60
    }
}

# -----------------------------
# Local mesh builder
# -----------------------------
def build_local_four_layer_mesh(n_donor, n_SC, n_EP, n_receptor):
    """
    Build a discontinuous-interface four-layer mesh.
    Adjacent layers have separate boundary nodes, and interface fluxes connect them.
    """
    layer_specs = [
        {
            "layer_id": 1,
            "layer_name": "Donor compartment",
            "short_name": "Donor",
            "length_cm": L1_DONOR_CM,
            "n_elements": n_donor
        },
        {
            "layer_id": 2,
            "layer_name": "Stratum corneum",
            "short_name": "SC",
            "length_cm": L2_SC_CM,
            "n_elements": n_SC
        },
        {
            "layer_id": 3,
            "layer_name": "Viable epidermis/dermis",
            "short_name": "EP/dermis",
            "length_cm": L3_EP_CM,
            "n_elements": n_EP
        },
        {
            "layer_id": 4,
            "layer_name": "Receptor compartment",
            "short_name": "Receptor",
            "length_cm": L4_RECEPTOR_CM,
            "n_elements": n_receptor
        }
    ]

    node_rows = []
    element_rows = []
    layer_rows = []

    global_node_id = 0
    global_element_id = 0
    x_global_start = 0.0
    boundary_nodes = {}

    for layer in layer_specs:
        layer_id = layer["layer_id"]
        length_cm = layer["length_cm"]
        n_elements = layer["n_elements"]
        h = length_cm / n_elements

        layer_start_cm = x_global_start
        layer_end_cm = x_global_start + length_cm

        first_node_id = global_node_id

        for local_node_id in range(n_elements + 1):
            x_local = local_node_id * h
            x_global = layer_start_cm + x_local

            node_rows.append({
                "global_node_id": global_node_id,
                "layer_id": layer_id,
                "layer_name": layer["layer_name"],
                "short_name": layer["short_name"],
                "local_node_id": local_node_id,
                "x_local_cm": x_local,
                "x_global_cm": x_global
            })

            global_node_id += 1

        last_node_id = global_node_id - 1

        boundary_nodes[layer_id] = {
            "left": first_node_id,
            "right": last_node_id
        }

        for local_element_id in range(n_elements):
            left_node = first_node_id + local_element_id
            right_node = left_node + 1

            element_rows.append({
                "global_element_id": global_element_id,
                "layer_id": layer_id,
                "layer_name": layer["layer_name"],
                "short_name": layer["short_name"],
                "local_element_id": local_element_id,
                "left_node": left_node,
                "right_node": right_node,
                "element_length_cm": h
            })

            global_element_id += 1

        layer_rows.append({
            "layer_id": layer_id,
            "layer_name": layer["layer_name"],
            "short_name": layer["short_name"],
            "length_cm": length_cm,
            "n_elements": n_elements,
            "start_cm": layer_start_cm,
            "end_cm": layer_end_cm
        })

        x_global_start = layer_end_cm

    nodes_df = pd.DataFrame(node_rows)
    elements_df = pd.DataFrame(element_rows)
    layers_df = pd.DataFrame(layer_rows)

    return nodes_df, elements_df, layers_df, boundary_nodes

# -----------------------------
# FEM matrix assembly
# -----------------------------
def assemble_local_fem_matrices(elements_df, n_total_nodes, D1, D2, D3, D4):
    """
    Assemble mass and diffusion matrices for a given mesh.
    """
    D_by_layer = {
        1: D1,
        2: D2,
        3: D3,
        4: D4
    }

    M = np.zeros((n_total_nodes, n_total_nodes), dtype=float)
    K = np.zeros((n_total_nodes, n_total_nodes), dtype=float)

    for _, element in elements_df.iterrows():
        i = int(element["left_node"])
        j = int(element["right_node"])
        h = float(element["element_length_cm"])
        layer_id = int(element["layer_id"])
        D = D_by_layer[layer_id]

        M_local = (h / 6.0) * np.array([
            [2.0, 1.0],
            [1.0, 2.0]
        ])

        K_local = (D / h) * np.array([
            [1.0, -1.0],
            [-1.0, 1.0]
        ])

        idx = [i, j]

        for a in range(2):
            for b in range(2):
                M[idx[a], idx[b]] += M_local[a, b]
                K[idx[a], idx[b]] += K_local[a, b]

    return M, K

# -----------------------------
# Layer amount integration
# -----------------------------
def compute_local_layer_amounts(C, elements_df):
    """
    Compute amount per area in each layer by trapezoidal integration.
    Unit: ug/cm2
    """
    rows = []

    for layer_id in [1, 2, 3, 4]:
        df_layer = elements_df[elements_df["layer_id"] == layer_id]

        amount = 0.0

        for _, element in df_layer.iterrows():
            i = int(element["left_node"])
            j = int(element["right_node"])
            h = float(element["element_length_cm"])
            amount += h * 0.5 * (C[i] + C[j])

        short_name = df_layer["short_name"].iloc[0]

        rows.append({
            "layer_id": layer_id,
            "short_name": short_name,
            "computed_amount_ug_cm2": amount
        })

    return pd.DataFrame(rows)

# -----------------------------
# Interface fluxes
# -----------------------------
def compute_local_interface_fluxes(C, boundary_nodes, P12, P23, P34, K34):
    """
    Positive flux means transport from donor to receptor direction.
    """
    C1_right = C[boundary_nodes[1]["right"]]
    C2_left = C[boundary_nodes[2]["left"]]

    C2_right = C[boundary_nodes[2]["right"]]
    C3_left = C[boundary_nodes[3]["left"]]

    C3_right = C[boundary_nodes[3]["right"]]
    C4_left = C[boundary_nodes[4]["left"]]

    J12 = K12_FIXED_CM_H * ((C1_right / P12) - C2_left)
    J23 = K23_FIXED_CM_H * ((C2_right / P23) - C3_left)
    J34 = K34 * ((C3_right / P34) - C4_left)

    return J12, J23, J34

# -----------------------------
# Local simulation function
# -----------------------------
def simulate_local_four_layer_model(
    times_h,
    mesh_config,
    Q0_eff_ug_cm2,
    D2_cm2_h,
    D3_cm2_h,
    P12,
    P23,
    P34,
    K34_cm_h
):
    """
    Simulate selected final model on a specified mesh.
    """
    nodes_df, elements_df, layers_df, boundary_nodes = build_local_four_layer_mesh(
        n_donor=mesh_config["n_donor"],
        n_SC=mesh_config["n_SC"],
        n_EP=mesh_config["n_EP"],
        n_receptor=mesh_config["n_receptor"]
    )

    n_total_nodes = len(nodes_df)

    M, K = assemble_local_fem_matrices(
        elements_df=elements_df,
        n_total_nodes=n_total_nodes,
        D1=D1_FIXED_CM2_H,
        D2=D2_cm2_h,
        D3=D3_cm2_h,
        D4=D4_FIXED_CM2_H
    )

    C0 = np.zeros(n_total_nodes, dtype=float)

    donor_nodes = nodes_df.loc[
        nodes_df["layer_id"] == 1,
        "global_node_id"
    ].astype(int).tolist()

    C0_donor = Q0_eff_ug_cm2 / L1_DONOR_CM
    C0[donor_nodes] = C0_donor

    def ode_system(t, C):
        rhs = -K @ C

        J12, J23, J34 = compute_local_interface_fluxes(
            C=C,
            boundary_nodes=boundary_nodes,
            P12=P12,
            P23=P23,
            P34=P34,
            K34=K34_cm_h
        )

        rhs[boundary_nodes[1]["right"]] -= J12
        rhs[boundary_nodes[2]["left"]] += J12

        rhs[boundary_nodes[2]["right"]] -= J23
        rhs[boundary_nodes[3]["left"]] += J23

        rhs[boundary_nodes[3]["right"]] -= J34
        rhs[boundary_nodes[4]["left"]] += J34

        dCdt = np.linalg.solve(M, rhs)

        return dCdt

    times_h = np.asarray(times_h, dtype=float)
    times_h = np.sort(np.unique(times_h))

    sol = solve_ivp(
        fun=ode_system,
        t_span=(0.0, float(times_h.max())),
        y0=C0,
        t_eval=times_h,
        method="LSODA",
        rtol=1e-7,
        atol=1e-9
    )

    if not sol.success:
        raise RuntimeError(f"Simulation failed: {sol.message}")

    amount_rows = []

    for idx, time_h in enumerate(sol.t):
        C_t = sol.y[:, idx]
        amount_df = compute_local_layer_amounts(C_t, elements_df)

        Q1 = amount_df.loc[amount_df["layer_id"] == 1, "computed_amount_ug_cm2"].iloc[0]
        Q2 = amount_df.loc[amount_df["layer_id"] == 2, "computed_amount_ug_cm2"].iloc[0]
        Q3 = amount_df.loc[amount_df["layer_id"] == 3, "computed_amount_ug_cm2"].iloc[0]
        Q4 = amount_df.loc[amount_df["layer_id"] == 4, "computed_amount_ug_cm2"].iloc[0]

        amount_rows.append({
            "time_h": float(time_h),
            "Q1_donor_ug_cm2": Q1,
            "Q2_SC_ug_cm2": Q2,
            "Q3_EP_ug_cm2": Q3,
            "Q4_receptor_ug_cm2": Q4,
            "Q_total_ug_cm2": Q1 + Q2 + Q3 + Q4
        })

    amount_df = pd.DataFrame(amount_rows)

    return amount_df, sol, nodes_df, elements_df, layers_df

# -----------------------------
# Run mesh independence simulations
# -----------------------------
mesh_times_h = np.array([0, 4, 6, 8, 12, 24], dtype=float)

mesh_result_rows = []
mesh_timecourse_rows = []

for _, param_row in selected_params_df.iterrows():
    formulation = param_row["formulation"]
    model_case = param_row["model_case"]

    print("\nRunning mesh test for:", formulation, "|", model_case)

    for mesh_name, mesh_config in mesh_cases.items():
        print("  Mesh:", mesh_name, mesh_config)

        sim_df, sol, nodes_df, elements_df, layers_df = simulate_local_four_layer_model(
            times_h=mesh_times_h,
            mesh_config=mesh_config,
            Q0_eff_ug_cm2=float(param_row["Q0_eff_ug_cm2"]),
            D2_cm2_h=float(param_row["D2_SC_cm2_h"]),
            D3_cm2_h=float(param_row["D3_EP_cm2_h"]),
            P12=float(param_row["P12_donor_SC"]),
            P23=float(param_row["P23_SC_EP"]),
            P34=float(param_row["P34_EP_receptor"]),
            K34_cm_h=float(param_row["K34_cm_h"])
        )

        sim_df["formulation"] = formulation
        sim_df["model_case"] = model_case
        sim_df["mesh_name"] = mesh_name
        sim_df["n_total_elements"] = sum(mesh_config.values())
        sim_df["n_total_nodes"] = len(nodes_df)

        mesh_timecourse_rows.append(sim_df)

        final_row = sim_df[sim_df["time_h"] == 24.0].iloc[0]
        initial_total = sim_df["Q_total_ug_cm2"].iloc[0]
        final_total = sim_df["Q_total_ug_cm2"].iloc[-1]

        mesh_result_rows.append({
            "formulation": formulation,
            "model_case": model_case,
            "mesh_name": mesh_name,
            "n_donor": mesh_config["n_donor"],
            "n_SC": mesh_config["n_SC"],
            "n_EP": mesh_config["n_EP"],
            "n_receptor": mesh_config["n_receptor"],
            "n_total_elements": sum(mesh_config.values()),
            "n_total_nodes": len(nodes_df),
            "Q_receptor_24h_ug_cm2": final_row["Q4_receptor_ug_cm2"],
            "Q_SC_24h_ug_cm2": final_row["Q2_SC_ug_cm2"],
            "Q_EP_24h_ug_cm2": final_row["Q3_EP_ug_cm2"],
            "Q_total_initial_ug_cm2": initial_total,
            "Q_total_final_ug_cm2": final_total,
            "mass_error_ug_cm2": final_total - initial_total
        })

mesh_independence_summary_df = pd.DataFrame(mesh_result_rows)
mesh_independence_timecourse_df = pd.concat(mesh_timecourse_rows, ignore_index=True)

# -----------------------------
# Compare all meshes against Fine mesh
# -----------------------------
comparison_rows = []

for formulation in selected_params_df["formulation"]:
    fine_row = mesh_independence_summary_df[
        (mesh_independence_summary_df["formulation"] == formulation)
        & (mesh_independence_summary_df["mesh_name"] == "Fine")
    ].iloc[0]

    for _, row in mesh_independence_summary_df[
        mesh_independence_summary_df["formulation"] == formulation
    ].iterrows():

        for output_name in [
            "Q_receptor_24h_ug_cm2",
            "Q_SC_24h_ug_cm2",
            "Q_EP_24h_ug_cm2"
        ]:
            fine_value = fine_row[output_name]
            mesh_value = row[output_name]

            relative_error_percent = (
                100.0 * abs(mesh_value - fine_value) / max(abs(fine_value), 1e-12)
            )

            comparison_rows.append({
                "formulation": formulation,
                "mesh_name": row["mesh_name"],
                "output": output_name,
                "mesh_value": mesh_value,
                "fine_mesh_value": fine_value,
                "absolute_difference": mesh_value - fine_value,
                "relative_difference_percent": relative_error_percent
            })

mesh_independence_comparison_df = pd.DataFrame(comparison_rows)

print("\nMesh independence summary:")
display(mesh_independence_summary_df)

print("\nMesh independence comparison against Fine mesh:")
display(mesh_independence_comparison_df)

# -----------------------------
# Save tables
# -----------------------------
summary_path = f"{tables_dir}/mesh_independence_summary.csv"
comparison_path = f"{tables_dir}/mesh_independence_comparison_against_fine.csv"
timecourse_path = f"{models_dir}/mesh_independence_timecourses.csv"

mesh_independence_summary_df.to_csv(summary_path, index=False)
mesh_independence_comparison_df.to_csv(comparison_path, index=False)
mesh_independence_timecourse_df.to_csv(timecourse_path, index=False)

print("\nSaved mesh independence tables:")
print(summary_path)
print(comparison_path)
print(timecourse_path)

# -----------------------------
# Plot receptor time-course across mesh cases
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)

for ax, formulation in zip(axes, ["Free KC", "KLZ-NPs"]):

    df_form = mesh_independence_timecourse_df[
        mesh_independence_timecourse_df["formulation"] == formulation
    ].copy()

    for mesh_name in ["Coarse", "Base", "Fine"]:
        df_mesh = df_form[df_form["mesh_name"] == mesh_name].sort_values("time_h")

        ax.plot(
            df_mesh["time_h"],
            df_mesh["Q4_receptor_ug_cm2"],
            marker="o",
            linewidth=2,
            label=mesh_name
        )

    ax.set_title(f"{formulation}: receptor prediction by mesh")
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Receptor amount (µg/cm²)")
    ax.legend(frameon=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle("Mesh independence test: receptor time-course prediction", y=1.03)
plt.tight_layout()

mesh_receptor_png = f"{figures_dir}/mesh_independence_receptor_timecourse.png"
mesh_receptor_pdf = f"{figures_dir}/mesh_independence_receptor_timecourse.pdf"
mesh_receptor_svg = f"{figures_dir}/mesh_independence_receptor_timecourse.svg"

plt.savefig(mesh_receptor_png, dpi=600, bbox_inches="tight")
plt.savefig(mesh_receptor_pdf, bbox_inches="tight")
plt.savefig(mesh_receptor_svg, bbox_inches="tight")

plt.show()

print("Saved mesh receptor time-course figure:")
print(mesh_receptor_png)
print(mesh_receptor_pdf)
print(mesh_receptor_svg)

# -----------------------------
# Plot 24 h endpoint differences against Fine mesh
# -----------------------------
plot_df = mesh_independence_comparison_df[
    mesh_independence_comparison_df["mesh_name"].isin(["Coarse", "Base"])
].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, formulation in zip(axes, ["Free KC", "KLZ-NPs"]):
    df_form = plot_df[plot_df["formulation"] == formulation].copy()

    labels = []
    values = []

    for mesh_name in ["Coarse", "Base"]:
        for output in [
            "Q_receptor_24h_ug_cm2",
            "Q_SC_24h_ug_cm2",
            "Q_EP_24h_ug_cm2"
        ]:
            row = df_form[
                (df_form["mesh_name"] == mesh_name)
                & (df_form["output"] == output)
            ].iloc[0]

            short_output = output.replace("Q_", "").replace("_24h_ug_cm2", "")
            labels.append(f"{mesh_name}\n{short_output}")
            values.append(row["relative_difference_percent"])

    ax.bar(np.arange(len(values)), values)
    ax.set_xticks(np.arange(len(values)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_title(formulation)
    ax.set_ylabel("Difference from Fine mesh (%)")
    ax.axhline(5.0, linestyle="--", linewidth=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle("Mesh sensitivity of 24 h compartment outputs", y=1.03)
plt.tight_layout()

mesh_endpoint_png = f"{figures_dir}/mesh_independence_24h_endpoint_sensitivity.png"
mesh_endpoint_pdf = f"{figures_dir}/mesh_independence_24h_endpoint_sensitivity.pdf"
mesh_endpoint_svg = f"{figures_dir}/mesh_independence_24h_endpoint_sensitivity.svg"

plt.savefig(mesh_endpoint_png, dpi=600, bbox_inches="tight")
plt.savefig(mesh_endpoint_pdf, bbox_inches="tight")
plt.savefig(mesh_endpoint_svg, bbox_inches="tight")

plt.show()

print("Saved mesh endpoint sensitivity figure:")
print(mesh_endpoint_png)
print(mesh_endpoint_pdf)
print(mesh_endpoint_svg)

# -----------------------------
# ZIP outputs
# -----------------------------
zip_path = f"{figures_dir}/mesh_independence_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in [
        summary_path,
        comparison_path,
        timecourse_path,
        mesh_receptor_png,
        mesh_receptor_pdf,
        mesh_receptor_svg,
        mesh_endpoint_png,
        mesh_endpoint_pdf,
        mesh_endpoint_svg
    ]:
        if os.path.exists(file_path):
            zipf.write(file_path, arcname=os.path.basename(file_path))

print("\nCreated ZIP file:")
print(zip_path)



In [ ]:
# Cell 17: Endpoint compartment balance table and figure at 24 h

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -----------------------------
# Use repository-relative project path if needed
# -----------------------------


models_dir = f"{PROJECT_DIR}/outputs/models"
figures_dir = f"{PROJECT_DIR}/outputs/figures"
tables_dir = f"{PROJECT_DIR}/outputs/tables"

os.makedirs(models_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)
os.makedirs(tables_dir, exist_ok=True)

# -----------------------------
# Experimental constants
# -----------------------------
AREA_CM2 = 2.2
DONOR_VOLUME_ML = 1.0
RECEPTOR_VOLUME_ML = 6.5

SKIN_THICKNESS_CM = 0.09
SC_THICKNESS_CM = 0.002
EP_THICKNESS_CM = SKIN_THICKNESS_CM - SC_THICKNESS_CM

L1_DONOR_CM = DONOR_VOLUME_ML / AREA_CM2
L2_SC_CM = SC_THICKNESS_CM
L3_EP_CM = EP_THICKNESS_CM
L4_RECEPTOR_CM = RECEPTOR_VOLUME_ML / AREA_CM2

layer_volume_info = {
    "Donor": {
        "amount_column": "Q1_donor_ug_cm2",
        "equivalent_thickness_cm": L1_DONOR_CM,
        "display_name": "Donor"
    },
    "SC": {
        "amount_column": "Q2_SC_ug_cm2",
        "equivalent_thickness_cm": L2_SC_CM,
        "display_name": "SC"
    },
    "EP/dermis": {
        "amount_column": "Q3_EP_ug_cm2",
        "equivalent_thickness_cm": L3_EP_CM,
        "display_name": "EP/dermis"
    },
    "Receptor": {
        "amount_column": "Q4_receptor_ug_cm2",
        "equivalent_thickness_cm": L4_RECEPTOR_CM,
        "display_name": "Receptor"
    }
}

# -----------------------------
# Load selected final model simulation
# -----------------------------
selected_simulation_path = f"{models_dir}/selected_final_model_dense_simulation.csv"

if not os.path.exists(selected_simulation_path):
    raise FileNotFoundError(
        f"Missing file: {selected_simulation_path}. Please run Cell 12 first."
    )

selected_simulation_df = pd.read_csv(selected_simulation_path)

print("Selected final model dense simulation loaded:")
display(selected_simulation_df.head())

# -----------------------------
# Extract 24 h endpoint rows
# -----------------------------
endpoint_rows = []

for formulation in ["Free KC", "KLZ-NPs"]:
    df_form = selected_simulation_df[
        selected_simulation_df["formulation"] == formulation
    ].copy()

    df_form = df_form.sort_values("time_h")

    # Use interpolation in case 24 h is not exactly present.
    model_case = df_form["model_case"].iloc[0]

    values_24h = {
        "formulation": formulation,
        "model_case": model_case,
        "time_h": 24.0
    }

    for compartment, info in layer_volume_info.items():
        q_value = np.interp(
            24.0,
            df_form["time_h"],
            df_form[info["amount_column"]]
        )

        concentration_value = q_value / info["equivalent_thickness_cm"]

        values_24h[f"{compartment}_amount_ug_cm2"] = q_value
        values_24h[f"{compartment}_average_concentration_ug_cm3"] = concentration_value

    total_amount = (
        values_24h["Donor_amount_ug_cm2"]
        + values_24h["SC_amount_ug_cm2"]
        + values_24h["EP/dermis_amount_ug_cm2"]
        + values_24h["Receptor_amount_ug_cm2"]
    )

    values_24h["total_model_amount_ug_cm2"] = total_amount

    for compartment in ["Donor", "SC", "EP/dermis", "Receptor"]:
        values_24h[f"{compartment}_percent_of_total"] = (
            100.0 * values_24h[f"{compartment}_amount_ug_cm2"] / total_amount
        )

    endpoint_rows.append(values_24h)

endpoint_balance_wide_df = pd.DataFrame(endpoint_rows)

print("Endpoint compartment balance, wide format:")
display(endpoint_balance_wide_df)

# -----------------------------
# Build long table for manuscript
# -----------------------------
long_rows = []

for _, row in endpoint_balance_wide_df.iterrows():
    for compartment in ["Donor", "SC", "EP/dermis", "Receptor"]:
        long_rows.append({
            "formulation": row["formulation"],
            "model_case": row["model_case"],
            "time_h": row["time_h"],
            "compartment": compartment,
            "model_amount_ug_cm2": row[f"{compartment}_amount_ug_cm2"],
            "model_average_concentration_ug_cm3": row[f"{compartment}_average_concentration_ug_cm3"],
            "percent_of_total_model_amount": row[f"{compartment}_percent_of_total"]
        })

endpoint_balance_long_df = pd.DataFrame(long_rows)

# Add note for donor.
endpoint_balance_long_df["interpretation_note"] = ""
endpoint_balance_long_df.loc[
    endpoint_balance_long_df["compartment"] == "Donor",
    "interpretation_note"
] = "Model-derived effective donor pool, not experimental donor recovery."

print("Endpoint compartment balance, manuscript-style long format:")
display(endpoint_balance_long_df)

# -----------------------------
# Add experimental endpoint comparison for measured compartments
# -----------------------------
# Try to load fitting predictions, because they contain experimental and model values
# for receptor, SC, and EP/dermis at 24 h.

selected_predictions_path = f"{models_dir}/selected_final_model_predictions.csv"

endpoint_experimental_comparison_df = None

if os.path.exists(selected_predictions_path):
    selected_predictions_df = pd.read_csv(selected_predictions_path)

    endpoint_experimental_comparison_df = selected_predictions_df[
        (
            (selected_predictions_df["compartment"].isin(["R", "SC", "EP"]))
            & (selected_predictions_df["time_h"] == 24.0)
        )
    ].copy()

    compartment_map = {
        "R": "Receptor",
        "SC": "SC",
        "EP": "EP/dermis"
    }

    endpoint_experimental_comparison_df["compartment_display"] = (
        endpoint_experimental_comparison_df["compartment"].map(compartment_map)
    )

    endpoint_experimental_comparison_df = endpoint_experimental_comparison_df[
        [
            "formulation",
            "model_case",
            "compartment_display",
            "time_h",
            "observed_ug_cm2",
            "model_ug_cm2",
            "error_ug_cm2"
        ]
    ].rename(columns={
        "compartment_display": "compartment",
        "observed_ug_cm2": "experimental_amount_ug_cm2",
        "model_ug_cm2": "model_amount_ug_cm2"
    })

    print("Experimental vs model endpoint comparison for measured compartments:")
    display(endpoint_experimental_comparison_df)

else:
    print("selected_final_model_predictions.csv not found. Experimental endpoint comparison skipped.")

# -----------------------------
# Save tables
# -----------------------------
endpoint_wide_path = f"{tables_dir}/endpoint_compartment_balance_24h_wide.csv"
endpoint_long_path = f"{tables_dir}/endpoint_compartment_balance_24h_long.csv"
endpoint_exp_comp_path = f"{tables_dir}/endpoint_experimental_vs_model_24h_measured_compartments.csv"

endpoint_balance_wide_df.to_csv(endpoint_wide_path, index=False)
endpoint_balance_long_df.to_csv(endpoint_long_path, index=False)

if endpoint_experimental_comparison_df is not None:
    endpoint_experimental_comparison_df.to_csv(endpoint_exp_comp_path, index=False)

print("Saved endpoint tables:")
print(endpoint_wide_path)
print(endpoint_long_path)
if endpoint_experimental_comparison_df is not None:
    print(endpoint_exp_comp_path)

# -----------------------------
# Figure 17A: Model endpoint amount distribution
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)

compartment_order = ["Donor", "SC", "EP/dermis", "Receptor"]

for ax, formulation in zip(axes, ["Free KC", "KLZ-NPs"]):
    df_plot = endpoint_balance_long_df[
        endpoint_balance_long_df["formulation"] == formulation
    ].copy()

    df_plot["compartment"] = pd.Categorical(
        df_plot["compartment"],
        categories=compartment_order,
        ordered=True
    )

    df_plot = df_plot.sort_values("compartment")

    x = np.arange(len(df_plot))

    ax.bar(
        x,
        df_plot["model_amount_ug_cm2"]
    )

    ax.set_xticks(x)
    ax.set_xticklabels(df_plot["compartment"], rotation=30, ha="right")
    ax.set_title(f"{formulation}: model compartment amounts at 24 h")
    ax.set_ylabel("Ketoconazole amount (µg/cm²)")
    ax.set_yscale("log")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle(
    "Model-predicted 24 h compartment amount distribution",
    y=1.03
)

plt.tight_layout()

endpoint_amount_png = f"{figures_dir}/endpoint_compartment_amount_distribution_24h.png"
endpoint_amount_pdf = f"{figures_dir}/endpoint_compartment_amount_distribution_24h.pdf"
endpoint_amount_svg = f"{figures_dir}/endpoint_compartment_amount_distribution_24h.svg"

plt.savefig(endpoint_amount_png, dpi=600, bbox_inches="tight")
plt.savefig(endpoint_amount_pdf, bbox_inches="tight")
plt.savefig(endpoint_amount_svg, bbox_inches="tight")

plt.show()

print("Saved endpoint amount distribution figure:")
print(endpoint_amount_png)
print(endpoint_amount_pdf)
print(endpoint_amount_svg)

# -----------------------------
# Figure 17B: Model endpoint average concentration distribution
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=False)

for ax, formulation in zip(axes, ["Free KC", "KLZ-NPs"]):
    df_plot = endpoint_balance_long_df[
        endpoint_balance_long_df["formulation"] == formulation
    ].copy()

    df_plot["compartment"] = pd.Categorical(
        df_plot["compartment"],
        categories=compartment_order,
        ordered=True
    )

    df_plot = df_plot.sort_values("compartment")

    x = np.arange(len(df_plot))

    ax.bar(
        x,
        df_plot["model_average_concentration_ug_cm3"]
    )

    ax.set_xticks(x)
    ax.set_xticklabels(df_plot["compartment"], rotation=30, ha="right")
    ax.set_title(f"{formulation}: average concentration at 24 h")
    ax.set_ylabel("Average concentration (µg/cm³)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle(
    "Model-predicted 24 h average concentration by compartment",
    y=1.03
)

plt.tight_layout()

endpoint_conc_png = f"{figures_dir}/endpoint_compartment_average_concentration_24h.png"
endpoint_conc_pdf = f"{figures_dir}/endpoint_compartment_average_concentration_24h.pdf"
endpoint_conc_svg = f"{figures_dir}/endpoint_compartment_average_concentration_24h.svg"

plt.savefig(endpoint_conc_png, dpi=600, bbox_inches="tight")
plt.savefig(endpoint_conc_pdf, bbox_inches="tight")
plt.savefig(endpoint_conc_svg, bbox_inches="tight")

plt.show()

print("Saved endpoint average concentration figure:")
print(endpoint_conc_png)
print(endpoint_conc_pdf)
print(endpoint_conc_svg)

# -----------------------------
# Figure 17C: Experimental vs model endpoint for measured compartments
# -----------------------------
if endpoint_experimental_comparison_df is not None:

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=False)

    measured_order = ["SC", "EP/dermis", "Receptor"]

    for ax, formulation in zip(axes, ["Free KC", "KLZ-NPs"]):
        df_plot = endpoint_experimental_comparison_df[
            endpoint_experimental_comparison_df["formulation"] == formulation
        ].copy()

        df_plot["compartment"] = pd.Categorical(
            df_plot["compartment"],
            categories=measured_order,
            ordered=True
        )

        df_plot = df_plot.sort_values("compartment")

        x = np.arange(len(df_plot))
        width = 0.35

        ax.bar(
            x - width / 2,
            df_plot["experimental_amount_ug_cm2"],
            width=width,
            label="Experimental"
        )

        ax.bar(
            x + width / 2,
            df_plot["model_amount_ug_cm2"],
            width=width,
            label="Model"
        )

        ax.set_xticks(x)
        ax.set_xticklabels(df_plot["compartment"], rotation=25, ha="right")
        ax.set_title(f"{formulation}: measured endpoints at 24 h")
        ax.set_ylabel("Ketoconazole amount (µg/cm²)")
        ax.legend(frameon=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.suptitle(
        "Experimental and model-predicted 24 h measured compartment endpoints",
        y=1.03
    )

    plt.tight_layout()

    endpoint_exp_model_png = f"{figures_dir}/endpoint_experimental_vs_model_measured_compartments_24h.png"
    endpoint_exp_model_pdf = f"{figures_dir}/endpoint_experimental_vs_model_measured_compartments_24h.pdf"
    endpoint_exp_model_svg = f"{figures_dir}/endpoint_experimental_vs_model_measured_compartments_24h.svg"

    plt.savefig(endpoint_exp_model_png, dpi=600, bbox_inches="tight")
    plt.savefig(endpoint_exp_model_pdf, bbox_inches="tight")
    plt.savefig(endpoint_exp_model_svg, bbox_inches="tight")

    plt.show()

    print("Saved experimental vs model endpoint figure:")
    print(endpoint_exp_model_png)
    print(endpoint_exp_model_pdf)
    print(endpoint_exp_model_svg)

# -----------------------------
# ZIP outputs
# -----------------------------
zip_path = f"{figures_dir}/endpoint_compartment_outputs_24h.zip"

files_to_zip = [
    endpoint_wide_path,
    endpoint_long_path,
    endpoint_amount_png,
    endpoint_amount_pdf,
    endpoint_amount_svg,
    endpoint_conc_png,
    endpoint_conc_pdf,
    endpoint_conc_svg
]

if endpoint_experimental_comparison_df is not None:
    files_to_zip += [
        endpoint_exp_comp_path,
        endpoint_exp_model_png,
        endpoint_exp_model_pdf,
        endpoint_exp_model_svg
    ]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in files_to_zip:
        if os.path.exists(file_path):
            zipf.write(file_path, arcname=os.path.basename(file_path))

print("Created ZIP file:")
print(zip_path)



In [ ]:
# Cell 18: Manuscript-ready tables and figure registry

import os
import zipfile
import numpy as np
import pandas as pd


# -----------------------------
# Use repository-relative project path
# -----------------------------


models_dir = f"{PROJECT_DIR}/outputs/models"
figures_dir = f"{PROJECT_DIR}/outputs/figures"
tables_dir = f"{PROJECT_DIR}/outputs/tables"
manuscript_dir = f"{PROJECT_DIR}/outputs/manuscript_ready_outputs"

os.makedirs(manuscript_dir, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("Manuscript output folder:", manuscript_dir)

# -----------------------------
# Load key model outputs
# -----------------------------
selected_params_path = f"{models_dir}/selected_final_model_parameters_fixed_P23.csv"

if not os.path.exists(selected_params_path):
    selected_params_path = f"{models_dir}/selected_final_model_parameters.csv"

selected_params_df = pd.read_csv(selected_params_path)
selected_params_df["P23_SC_EP"] = selected_params_df["P23_SC_EP"].fillna(1.0)

selected_metrics_df = pd.read_csv(f"{models_dir}/selected_final_model_metrics.csv")
model_selection_df = pd.read_csv(f"{tables_dir}/selected_final_model_rationale.csv")

mesh_summary_df = pd.read_csv(f"{tables_dir}/mesh_independence_summary.csv")
mesh_comparison_df = pd.read_csv(f"{tables_dir}/mesh_independence_comparison_against_fine.csv")

endpoint_long_df = pd.read_csv(f"{tables_dir}/endpoint_compartment_balance_24h_long.csv")
endpoint_measured_df = pd.read_csv(f"{tables_dir}/endpoint_experimental_vs_model_24h_measured_compartments.csv")

# -----------------------------
# Table 1: Experimental constants
# -----------------------------
experimental_constants_df = pd.DataFrame([
    {
        "parameter": "Diffusion area",
        "symbol": "A",
        "value": 2.2,
        "unit": "cm^2",
        "note": "Franz diffusion cell exposure area"
    },
    {
        "parameter": "Donor volume",
        "symbol": "V_D",
        "value": 1.0,
        "unit": "mL",
        "note": "Used as equivalent computational donor thickness"
    },
    {
        "parameter": "Receptor volume",
        "symbol": "V_R",
        "value": 6.5,
        "unit": "mL",
        "note": "Used to convert receptor amount to receptor concentration"
    },
    {
        "parameter": "Total skin thickness",
        "symbol": "L_skin",
        "value": 0.09,
        "unit": "cm",
        "note": "Model skin thickness"
    },
    {
        "parameter": "Stratum corneum thickness",
        "symbol": "L_SC",
        "value": 0.002,
        "unit": "cm",
        "note": "20 micrometers"
    },
    {
        "parameter": "EP/dermis thickness",
        "symbol": "L_EP",
        "value": 0.088,
        "unit": "cm",
        "note": "Total skin thickness minus SC thickness"
    },
    {
        "parameter": "Temperature",
        "symbol": "T",
        "value": 32.0,
        "unit": "degree C",
        "note": "Experimental skin surface temperature"
    }
])

# -----------------------------
# Table 2: Selected final model parameters
# -----------------------------
parameter_name_map = {
    "Q0_eff_ug_cm2": "Effective donor loading",
    "D2_SC_cm2_h": "SC diffusion coefficient",
    "D3_EP_cm2_h": "EP/dermis diffusion coefficient",
    "P12_donor_SC": "Donor-SC partition coefficient",
    "P23_SC_EP": "SC-EP/dermis partition coefficient",
    "P34_EP_receptor": "EP/dermis-receptor partition coefficient",
    "K34_cm_h": "EP/dermis-receptor mass transfer coefficient"
}

parameter_unit_map = {
    "Q0_eff_ug_cm2": "ug/cm^2",
    "D2_SC_cm2_h": "cm^2/h",
    "D3_EP_cm2_h": "cm^2/h",
    "P12_donor_SC": "-",
    "P23_SC_EP": "-",
    "P34_EP_receptor": "-",
    "K34_cm_h": "cm/h"
}

parameter_rows = []

for _, row in selected_params_df.iterrows():
    for parameter_code in parameter_name_map:
        parameter_rows.append({
            "formulation": row["formulation"],
            "selected_model": row["model_case"],
            "parameter": parameter_name_map[parameter_code],
            "parameter_code": parameter_code,
            "value": row[parameter_code],
            "unit": parameter_unit_map[parameter_code]
        })

manuscript_parameter_table_df = pd.DataFrame(parameter_rows)

# -----------------------------
# Table 3: Selected final model performance
# -----------------------------
manuscript_performance_table_df = selected_metrics_df.copy()

manuscript_performance_table_df = manuscript_performance_table_df[
    [
        "formulation",
        "model_case",
        "target",
        "RMSE_ug_cm2",
        "MAE_ug_cm2",
        "R2",
        "n"
    ]
].copy()

# -----------------------------
# Table 4: Model selection rationale
# -----------------------------
manuscript_model_selection_table_df = model_selection_df.copy()

# -----------------------------
# Table 5: Mesh independence summary
# Keep only the main 24 h outputs.
# -----------------------------
mesh_main_outputs_df = mesh_comparison_df[
    mesh_comparison_df["mesh_name"].isin(["Coarse", "Base"])
].copy()

mesh_main_outputs_df = mesh_main_outputs_df[
    [
        "formulation",
        "mesh_name",
        "output",
        "mesh_value",
        "fine_mesh_value",
        "relative_difference_percent"
    ]
].copy()

# -----------------------------
# Table 6: Endpoint measured compartment comparison
# -----------------------------
manuscript_endpoint_table_df = endpoint_measured_df.copy()

manuscript_endpoint_table_df = manuscript_endpoint_table_df[
    [
        "formulation",
        "model_case",
        "compartment",
        "time_h",
        "experimental_amount_ug_cm2",
        "model_amount_ug_cm2",
        "error_ug_cm2"
    ]
].copy()

# -----------------------------
# Figure registry
# -----------------------------
figure_registry_rows = [
    {
        "figure_label": "Figure 2",
        "recommended_location": "Main text",
        "figure_title": "Experimental receptor permeation and skin-retention profiles",
        "file_png": "outputs/figures/figure_experimental_input_profiles.png",
        "purpose": "Digitized receptor, SC, and EP/dermis profiles used in the analysis."
    },
    {
        "figure_label": "Figure 3A-B",
        "recommended_location": "Main text",
        "figure_title": "Selected receptor-time fits",
        "file_png": "outputs/figures/case1_case2_receptor_fit_comparison.png",
        "purpose": "Selected Free KC and KLZ-NP receptor-time fits."
    },
    {
        "figure_label": "Figure 3C-D / Figure S2",
        "recommended_location": "Main text and Supplementary",
        "figure_title": "Experimental versus model-predicted 24 h measured compartments",
        "file_png": "outputs/figures/endpoint_experimental_vs_model_measured_compartments_24h.png",
        "purpose": "Calibration endpoint comparison for SC, EP/dermis, and receptor."
    },
    {
        "figure_label": "Figure 4",
        "recommended_location": "Main text",
        "figure_title": "Selected skin-layer concentration profiles",
        "file_png": "outputs/figures/figure4_FINAL_journal_ready_skin_layer_profiles.png",
        "purpose": "Model-derived spatial concentration profiles in SC and EP/dermis."
    },
    {
        "figure_label": "Figure S1",
        "recommended_location": "Supplementary",
        "figure_title": "Mesh sensitivity of 24 h endpoints",
        "file_png": "outputs/figures/mesh_independence_24h_endpoint_sensitivity.png",
        "purpose": "Relative differences versus the fine mesh."
    },
    {
        "figure_label": "Figure S3a",
        "recommended_location": "Supplementary",
        "figure_title": "Case 1 receptor residuals",
        "file_png": "outputs/figures/case1_receptor_residuals.png",
        "purpose": "Residual diagnostic for the Case 1 fits."
    },
    {
        "figure_label": "Figure S3b",
        "recommended_location": "Supplementary",
        "figure_title": "Case 2 receptor residuals",
        "file_png": "outputs/figures/case2_receptor_residuals.png",
        "purpose": "Residual diagnostic for the Case 2 fits."
    },
    {
        "figure_label": "Figure S4",
        "recommended_location": "Supplementary",
        "figure_title": "Full computational-domain concentration profiles",
        "file_png": "outputs/figures/supplementary_full_computational_domain_concentration_profiles.png",
        "purpose": "Model-derived profiles across donor, SC, EP/dermis, and receptor domains."
    },
    {
        "figure_label": "Figure S5",
        "recommended_location": "Supplementary",
        "figure_title": "Receptor concentration-time profiles",
        "file_png": "outputs/figures/receptor_concentration_time_profiles.png",
        "purpose": "Amount-to-concentration conversion for the receptor compartment."
    },
    {
        "figure_label": "Figure S6",
        "recommended_location": "Supplementary",
        "figure_title": "24 h compartment amount distribution",
        "file_png": "outputs/figures/endpoint_compartment_amount_distribution_24h.png",
        "purpose": "Model-predicted donor, SC, EP/dermis, and receptor amounts."
    },
    {
        "figure_label": "Figure S7",
        "recommended_location": "Supplementary",
        "figure_title": "24 h average concentration by compartment",
        "file_png": "outputs/figures/endpoint_compartment_average_concentration_24h.png",
        "purpose": "Average concentration normalized by compartment thickness."
    }
]

figure_registry_df = pd.DataFrame(figure_registry_rows)

# Check which figure files exist.
figure_registry_df["file_exists"] = figure_registry_df["file_png"].apply(lambda rel: (REPO_ROOT / rel).exists())

print("Figure registry:")
display(figure_registry_df)

# -----------------------------
# Save manuscript-ready tables
# -----------------------------
table_paths = {}

table_paths["table1_experimental_constants"] = f"{manuscript_dir}/Table_1_experimental_constants.csv"
table_paths["table2_selected_model_parameters"] = f"{manuscript_dir}/Table_2_selected_model_parameters.csv"
table_paths["table3_selected_model_performance"] = f"{manuscript_dir}/Table_3_selected_model_performance.csv"
table_paths["table4_model_selection_rationale"] = f"{manuscript_dir}/Table_4_model_selection_rationale.csv"
table_paths["tableS1_mesh_independence"] = f"{manuscript_dir}/Table_S1_mesh_independence.csv"
table_paths["tableS2_endpoint_comparison"] = f"{manuscript_dir}/Table_S2_endpoint_experimental_vs_model.csv"
table_paths["figure_registry"] = f"{manuscript_dir}/Figure_registry.csv"

experimental_constants_df.to_csv(table_paths["table1_experimental_constants"], index=False)
manuscript_parameter_table_df.to_csv(table_paths["table2_selected_model_parameters"], index=False)
manuscript_performance_table_df.to_csv(table_paths["table3_selected_model_performance"], index=False)
manuscript_model_selection_table_df.to_csv(table_paths["table4_model_selection_rationale"], index=False)
mesh_main_outputs_df.to_csv(table_paths["tableS1_mesh_independence"], index=False)
manuscript_endpoint_table_df.to_csv(table_paths["tableS2_endpoint_comparison"], index=False)
figure_registry_df.to_csv(table_paths["figure_registry"], index=False)

print("Saved manuscript-ready tables:")
for name, path in table_paths.items():
    print(name, ":", path)

# -----------------------------
# Create one Excel workbook with all manuscript tables
# -----------------------------
excel_path = f"{manuscript_dir}/manuscript_ready_tables.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    experimental_constants_df.to_excel(writer, sheet_name="Table 1 Constants", index=False)
    manuscript_parameter_table_df.to_excel(writer, sheet_name="Table 2 Parameters", index=False)
    manuscript_performance_table_df.to_excel(writer, sheet_name="Table 3 Performance", index=False)
    manuscript_model_selection_table_df.to_excel(writer, sheet_name="Table 4 Selection", index=False)
    mesh_main_outputs_df.to_excel(writer, sheet_name="Table S1 Mesh", index=False)
    manuscript_endpoint_table_df.to_excel(writer, sheet_name="Table S2 Endpoints", index=False)
    figure_registry_df.to_excel(writer, sheet_name="Figure Registry", index=False)

print("Saved Excel workbook:")
print(excel_path)

# -----------------------------
# Zip manuscript-ready outputs
# -----------------------------
zip_path = f"{manuscript_dir}/manuscript_ready_tables_and_registry.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for path in list(table_paths.values()) + [excel_path]:
        if os.path.exists(path):
            zipf.write(path, arcname=os.path.basename(path))

print("Created ZIP file:")
print(zip_path)



In [ ]:
# Cell 19: Build Figure 2 experimental input profiles

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -----------------------------
# Use repository-relative project path
# -----------------------------


models_dir = f"{PROJECT_DIR}/outputs/models"
figures_dir = f"{PROJECT_DIR}/outputs/figures"
tables_dir = f"{PROJECT_DIR}/outputs/tables"
data_processed_dir = f"{PROJECT_DIR}/data/processed"

os.makedirs(figures_dir, exist_ok=True)

# -----------------------------
# Load experimental data
# -----------------------------
receptor_path_candidates = [
    f"{data_processed_dir}/receptor_time_data.csv",
    f"{PROJECT_DIR}/receptor_time_data.csv",
    f"{PROJECT_DIR}/data/raw/receptor_time_data.csv"
]

skin_retention_path_candidates = [
    f"{data_processed_dir}/skin_retention_data_qualitative.csv",
    f"{PROJECT_DIR}/skin_retention_data_qualitative.csv",
    f"{PROJECT_DIR}/data/raw/skin_retention_data_qualitative.csv"
]

receptor_df = None
for path in receptor_path_candidates:
    if os.path.exists(path):
        receptor_df = pd.read_csv(path)
        print("Loaded receptor data from:", path)
        break

if receptor_df is None:
    raise FileNotFoundError("Could not find receptor_time_data.csv.")

skin_retention_df = None
for path in skin_retention_path_candidates:
    if os.path.exists(path):
        skin_retention_df = pd.read_csv(path)
        print("Loaded skin retention data from:", path)
        break

if skin_retention_df is None:
    raise FileNotFoundError("Could not find skin_retention_data_qualitative.csv.")

print("Receptor data:")
display(receptor_df)

print("Skin retention data:")
display(skin_retention_df.head())

# -----------------------------
# Standardize columns
# -----------------------------
# receptor_df expected:
# formulation, time_h, Q_receptor_ug_cm2

if "Q_receptor_ug_cm2" not in receptor_df.columns:
    raise KeyError(f"Q_receptor_ug_cm2 missing. Columns: {receptor_df.columns.tolist()}")

# skin_retention_df expected:
# formulation, compartment, time_h, Q_ug_cm2

if "Q_ug_cm2" not in skin_retention_df.columns:
    raise KeyError(f"Q_ug_cm2 missing. Columns: {skin_retention_df.columns.tolist()}")

# -----------------------------
# Build Figure 2
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))

formulations = ["Free KC", "KLZ-NPs"]

# Panel A: receptor permeation
ax = axes[0]

for formulation in formulations:
    df_plot = receptor_df[
        receptor_df["formulation"] == formulation
    ].sort_values("time_h")

    ax.plot(
        df_plot["time_h"],
        df_plot["Q_receptor_ug_cm2"],
        marker="o",
        linewidth=2,
        label=formulation
    )

ax.set_title("A. Receptor permeation")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Panel B: SC retention
ax = axes[1]

for formulation in formulations:
    df_plot = skin_retention_df[
        (skin_retention_df["formulation"] == formulation)
        & (skin_retention_df["compartment"] == "SC")
    ].sort_values("time_h")

    ax.plot(
        df_plot["time_h"],
        df_plot["Q_ug_cm2"],
        marker="o",
        linewidth=2,
        label=formulation
    )

ax.set_title("B. SC retention")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Panel C: EP/dermis retention
ax = axes[2]

for formulation in formulations:
    df_plot = skin_retention_df[
        (skin_retention_df["formulation"] == formulation)
        & (skin_retention_df["compartment"] == "EP")
    ].sort_values("time_h")

    ax.plot(
        df_plot["time_h"],
        df_plot["Q_ug_cm2"],
        marker="o",
        linewidth=2,
        label=formulation
    )

ax.set_title("C. EP/dermis retention")
ax.set_xlabel("Time (h)")
ax.set_ylabel("Ketoconazole amount (µg/cm²)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.suptitle(
    "Experimental ketoconazole permeation and skin-retention data",
    y=1.03
)

plt.tight_layout()

figure2_png = f"{figures_dir}/figure_experimental_input_profiles.png"
figure2_pdf = f"{figures_dir}/figure_experimental_input_profiles.pdf"
figure2_svg = f"{figures_dir}/figure_experimental_input_profiles.svg"

plt.savefig(figure2_png, dpi=600, bbox_inches="tight")
plt.savefig(figure2_pdf, bbox_inches="tight")
plt.savefig(figure2_svg, bbox_inches="tight")

plt.show()

print("Saved Figure 2 experimental input profiles:")
print(figure2_png)
print(figure2_pdf)
print(figure2_svg)

# -----------------------------
# Optional archive creation
# -----------------------------
zip_path = f"{figures_dir}/figure2_experimental_input_profiles.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for path in [figure2_png, figure2_pdf, figure2_svg]:
        if os.path.exists(path):
            zipf.write(path, arcname=os.path.basename(path))

print("Created ZIP file:")
print(zip_path)



In [ ]:
# Cell 19.1: Update and verify figure registry after creating Figure 2

import os
import pandas as pd



figures_dir = f"{PROJECT_DIR}/outputs/figures"
manuscript_dir = f"{PROJECT_DIR}/outputs/manuscript_ready_outputs"

figure_registry_path = f"{manuscript_dir}/Figure_registry.csv"

figure_registry_df = pd.read_csv(figure_registry_path)

# Refresh file existence check
figure_registry_df["file_exists"] = figure_registry_df["file_png"].apply(lambda rel: (REPO_ROOT / rel).exists())

# Save updated registry
figure_registry_df.to_csv(figure_registry_path, index=False)

print("Updated Figure Registry:")
display(figure_registry_df)

print("\nMissing figures:")
display(figure_registry_df[figure_registry_df["file_exists"] == False])